In [ ]:
### als version


import os, re, sys, math, glob, zipfile, shutil, cv2, inspect, time, random, openpyxl
import numpy as np
import pandas as pd
import tkinter as tk
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import spectral.io.envi as envi
import uuid 
from datetime import datetime
from PIL import Image
from pathlib import Path
from tkinter import ttk, filedialog, messagebox
from scipy.ndimage import zoom
from skimage.transform import AffineTransform, warp
from openpyxl.styles import Color
from openpyxl.formatting.rule import DataBarRule



# Theme path for GUI
THEME_PATH_FALLBACK = r"C:\Users\Aditya\Documents\python\ANC_processor\tkinter_themes\Azure-ttk-theme-main\azure.tcl"

# Transform parameters file for coregistration
TRANSFORM_PARAMETERS_FILE = r"D:\Filezilla\Coreg Essen\transform_parameters.txt"

# Model path for HSI masking
# MASK_MODEL_PATH = r"D:\Hellas Gold\mask_hellas.pth"
MASK_MODEL_PATH = r"D:\essen\05_06_2025_pth27.pth"


# Allow processing of very large images
Image.MAX_IMAGE_PIXELS = None

#  1) LARGER I/O BUFFERS FOR SPEED   
def open_with_buffer(filename, mode, buffering=2**20):
    """
    Same as open(filename, mode), but sets a large buffering (1 MB by default).
    Does not change any logic or outputs – only speeds up I/O.
    """
    return open(filename, mode, buffering=buffering)


#  Stdout Redirector with Real-Time Flush   
class TextRedirector:
    def __init__(self, widget, tag="stdout"):
        self.widget = widget
        self.tag = tag

    def write(self, s):
        self.widget.insert(tk.END, s)
        self.widget.see(tk.END)
        # Force immediate UI update so that each print is shown in real time:
        self.widget.update_idletasks()

    def flush(self):
        # Ensures Python won't buffer the prints:
        self.widget.update_idletasks()
        pass



#  CODE 1: Extraction Logic   

def parse_dt(val):
    val = val.strip().split(" ")[0]
    try:
        d = datetime.strptime(val, "%Y-%m-%dT%H:%M:%S.%f")
    except:
        d = datetime.strptime(val, "%Y-%m-%dT%H:%M:%S")
    return d.strftime("%Y-%m-%d"), d.strftime("%H:%M:%S")

def hsiDataPadder(raw_path, samples, lines, bands, data_type, interleave):
    if not os.path.isfile(raw_path):
        return
    dt_map = {"12": np.uint16, "4": np.float32, "2": np.int16}
    dt = dt_map.get(data_type, np.uint16)
    with open_with_buffer(raw_path, "rb") as f:
        arr = np.fromfile(f, dt)
    s = int(samples)
    l = int(lines)
    b = int(bands)
    if interleave.lower() == "bil":
        arr = arr.reshape((l, b, s))
        pad = np.zeros((l, b, 384), dt)
        off = (384 - s) // 2
        pad[:, :, off:off+s] = arr
    elif interleave.lower() == "bip":
        arr = arr.reshape((l, s, b))
        pad = np.zeros((l, 384, b), dt)
        off = (384 - s) // 2
        pad[:, off:off+s, :] = arr
    else:  # BSQ
        arr = arr.reshape((b, l, s))
        pad = np.zeros((b, l, 384), dt)
        off = (384 - s) // 2
        pad[:, :, off:off+s] = arr
    with open_with_buffer(raw_path, "wb") as f:
        pad.tofile(f)
    print(f"[PADDED] {raw_path}")
    sys.stdout.flush()  # force immediate flush of the print

def writeHdrFile(meta, data, hdr_path):
    metaOrder = [
        "Name","Company","Location","WellID","CoreRun","CoreBox","Date",
        "Depth From","Depth To","Length","Diameter","Slabbed","Geologist",
        "RecordedBy","Comment","DepthInEarth","DepthInRock","TotalDepth From",
        "BottomElevation","RockElevation","CoreRecovery","scan length[mm]",
        "Resolution","FramePeriod","IntegrationTime","CameraName","CameraSerial",
        "Measurement-ID","Line-ID"
    ]
    dataOrder = [
        "samples","lines","bands","data type","interleave","file type",
        "misc","byte order","acquisition date","acquisition time","wavelength"
    ]
    sc_len = meta.get("scan length[mm]", "").strip()
    if not sc_len:
        try:
            df = float(meta.get("Depth From","0"))
            dt = float(meta.get("Depth To","0"))
            meta["scan length[mm]"] = f"{(dt - df)*1000:.4f}"
        except:
            meta["scan length[mm]"] = "0.0"

    try:
        ln_val = int(data.get("lines","0"))
        s_len  = float(meta.get("scan length[mm]", "0"))
        if ln_val>0 and s_len>0:
            meta["Resolution"] = f"{ln_val/s_len:.4f} pixel/mm"
        else:
            meta["Resolution"] = "UNKNOWN"
    except:
        meta["Resolution"] = "UNKNOWN"

    data["data type"] = "12"  # force data type=12
    with open_with_buffer(hdr_path, "w") as fw:
        fw.write("ENVI\ndescription = {\n")
        for k in metaOrder:
            fw.write(f"{k} = {meta.get(k,'0.0')}\n")
        fw.write("}\n")
        for k in dataOrder:
            fw.write(f"{k} = {data.get(k,'BLANK')}\n")

def build_url_dict(root):
    url_map = {}
    blocks = []
    for tname in ["HSIresults","VNIRresults"]:
        b = root.find(f".//{tname}")
        if b is not None:
            blocks.extend(b.findall("HSIresult"))

    for res in blocks:
        line_id = res.findtext("lineId","NoLine")
        camera  = res.findtext("CameraName","")
        cam_ser = res.findtext("CameraSerial","")
        try:
            df = float(res.find(".//startTime/position").text.strip())
            dt = float(res.find(".//stopTime/position").text.strip())
        except:
            df, dt = 0.0, 0.0
        fp = res.findtext("FramePeriod_us","0")
        it = res.findtext("IntegrationTime_us","0")
        st = res.find(".//startTime/time")
        ad, at = ("2025-01-01","00:00:00")
        if st is not None and st.text:
            ad, at = parse_dt(st.text)
        img_node = res.find("Images/*")
        if img_node is None:
            continue
        url_str = img_node.findtext("Url","").lower()
        url_base = os.path.splitext(os.path.basename(url_str))[0]
        smp   = img_node.findtext("Samples","0")
        lns   = img_node.findtext("Lines","0")
        bnd   = img_node.findtext("Bands","0")
        dtyp  = img_node.findtext("DataType","12")
        ilv   = img_node.findtext("Interleave","bil")
        bo    = img_node.findtext("byteorder","0")

        wls = "{ }"
        wtag = img_node.find("Wavelengths")
        if wtag is not None and wtag.text:
            wtxt = wtag.text.replace(";", " ,")
            arr = [x.strip() for x in wtxt.split(",") if x.strip()]
            wls = "{ " + ", ".join(arr) + " }"

        url_map[url_base] = {
            'depth_from':   df,
            'depth_to':     dt,
            'camera':       camera,
            'camera_serial': cam_ser,
            'line_id':      line_id,
            'frameperiod':  fp,
            'integration':  it,
            'acq_date':     ad,
            'acq_time':     at,
            'samples':      smp,
            'lines':        lns,
            'bands':        bnd,
            'dtype':        dtyp,
            'ilv':          ilv,
            'b_order':      bo,
            'wls':          wls
        }
    return url_map

def create_hdrs_and_pad_optimized(xml_path, working_folder, sample_info):
    if not os.path.isfile(xml_path):
        return
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except:
        return

    url_dict = build_url_dict(root)
    for f in os.listdir(working_folder):
        if not f.lower().endswith(".raw"):
            continue
        raw_noext_lower = os.path.splitext(f)[0].lower()
        if raw_noext_lower not in url_dict:
            print(f"[WARN] No matching block for {f}")
            sys.stdout.flush()
            continue

        info = url_dict[raw_noext_lower]
        df, dt   = info['depth_from'], info['depth_to']
        camera   = info['camera']
        cam_ser  = info['camera_serial']
        line_id  = info['line_id']
        fp       = info['frameperiod']
        it       = info['integration']
        ad, at   = info['acq_date'], info['acq_time']
        smp      = info['samples']
        lns      = info['lines']
        bnd      = info['bands']
        dtyp     = info['dtype']
        ilv      = info['ilv']
        bo       = info['b_order']
        wls      = info['wls']

        raw_path = os.path.join(working_folder, f)
        hdr_path = os.path.join(working_folder, raw_noext_lower + ".hdr")

        meta = dict(sample_info)
        meta["Depth From"]       = f"{df:.6f}"
        meta["Depth To"]         = f"{dt:.6f}"
        meta["FramePeriod"]      = fp
        meta["IntegrationTime"]  = it
        meta["CameraName"]       = camera
        meta["CameraSerial"]     = cam_ser
        meta["Line-ID"]          = line_id

        data = {
            "samples": smp,
            "lines": lns,
            "bands": bnd,
            "data type": dtyp,
            "interleave": ilv,
            "file type": "ENVI",
            "misc": "empty",
            "byte order": bo,
            "acquisition date": ad,
            "acquisition time": at,
            "wavelength": wls
        }
        writeHdrFile(meta, data, hdr_path)

        # Pad if needed
        try:
            si_i = int(smp)
            if si_i < 384:
                hsiDataPadder(raw_path, smp, lns, bnd, dtyp, ilv)
        except:
            pass

        new_hdr_name = f"{df:.3f}_{dt:.3f}_{line_id}.hdr"
        new_raw_name = f"{df:.3f}_{dt:.3f}_{line_id}.raw"
        os.rename(hdr_path, os.path.join(working_folder, new_hdr_name))
        os.rename(raw_path, os.path.join(working_folder, new_raw_name))
        print(f"[PROCESSED] {f} => {new_hdr_name} / {new_raw_name}")
        sys.stdout.flush()

def extract_and_rename(base_folder):
    xml_dir  = os.path.join(base_folder,"XML")
    wr_dir   = os.path.join(base_folder,"Extracted_WR")
    rgb_dir  = os.path.join(base_folder,"RGB Core Box Images")
    swir_dir = os.path.join(base_folder,"SWIR_HSI_Files")
    vnir_dir = os.path.join(base_folder,"VNIR_HSI_Files")

    for d in [xml_dir, wr_dir, rgb_dir, swir_dir, vnir_dir]:
        os.makedirs(d, exist_ok=True)

    for file in os.listdir(base_folder):
        if not file.lower().endswith(".ancprj"):
            continue
        if "_processed" in file.lower():
            print(f"[SKIP] {file} => already processed.")
            sys.stdout.flush()
            continue

        ancprj_path = os.path.join(base_folder, file)
        base_noext  = os.path.splitext(file)[0]
        zip_name    = base_noext + ".zip"
        zip_path    = os.path.join(base_folder, zip_name)

        os.rename(ancprj_path, zip_path)
        print(f"Renamed {file} => {zip_name}")
        sys.stdout.flush()

        temp_folder = os.path.join(base_folder, base_noext + "_temp")
        os.makedirs(temp_folder, exist_ok=True)

        xml_extracted    = None
        meas_id_for_rgb  = "NoID"

        with zipfile.ZipFile(zip_path, "r") as zf:
            # 1) find the .xml to parse measurement ID
            for info in zf.infolist():
                lname = info.filename.lower()
                if "results" in lname and not ("swirimages" in lname or "vnirimages" in lname):
                    continue
                if lname.endswith(".xml"):
                    xdest = os.path.join(temp_folder, os.path.basename(info.filename))
                    # Speed up extraction with bigger buffering:
                    with zf.open(info) as src, open_with_buffer(xdest, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    xml_extracted = xdest
                    try:
                        dxml = ET.parse(xdest)
                        rxml = dxml.getroot()
                        meas_id_for_rgb = rxml.findtext(".//measurement/id", "NoID")
                    except:
                        pass

            # 2) extract raw / WR / RGB
            for info in zf.infolist():
                lname = info.filename.lower()
                if "results" in lname and not ("swirimages" in lname or "vnirimages" in lname):
                    continue
                if lname.endswith(".xml"):
                    continue
                if ("neoreftarget_scan.envi" in lname or "neoreftarget_scan.hdr" in lname):
                    suffix = ""
                    if "swirimages" in lname:
                        suffix = "_SWIR"
                    elif "vnirimages" in lname:
                        suffix = "_VNIR"
                    
                    wr_out = f"{base_noext}_NeoRefTarget_Scan{suffix}" + os.path.splitext(info.filename)[1]
                    with zf.open(info) as src, open_with_buffer(os.path.join(wr_dir, wr_out), "wb") as dst:
                        shutil.copyfileobj(src, dst)
                
                elif "referencepaneldefinition" in lname.lower() and ("swirimages" in lname or "vnirimages" in lname):
                    suffix = ""
                    if "swirimages" in lname:
                        suffix = "_SWIR"
                    elif "vnirimages" in lname:
                        suffix = "_VNIR"
                    
                    # Explicitly extract only to the Extracted_WR folder with a specific name
                    wr_out = f"{base_noext}_ReferencePanelDefinition{suffix}" + os.path.splitext(info.filename)[1]
                    wr_dest = os.path.join(wr_dir, wr_out)
                    
                    with zf.open(info) as src, open_with_buffer(wr_dest, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    
                    print(f"[EXTRACTED REFERENCE PANEL] => {wr_out} to {wr_dir}")
                    sys.stdout.flush()

                elif lname.endswith(".raw") and ("swirimages" in lname or "vnirimages" in lname) and "referencepaneldefinition" not in lname.lower():
                    out_raw = os.path.join(temp_folder, os.path.basename(info.filename))
                    with zf.open(info) as src, open_with_buffer(out_raw, "wb") as dst:
                        shutil.copyfileobj(src, dst)

                elif info.filename.endswith("RGBScanner.jpg"):
                    out_jpg = f"{base_noext}_{meas_id_for_rgb}.jpg"
                    with zf.open(info) as src, open_with_buffer(os.path.join(rgb_dir, out_jpg), "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    print(f"[EXTRACTED RGB] => {out_jpg}")
                    sys.stdout.flush()

        final_xml_path = None
        sample_info = {}
        depth_val = "NoDepth"
        real_mid = "NoID"

        if xml_extracted and os.path.exists(xml_extracted):
            try:
                doc = ET.parse(xml_extracted)
                rt  = doc.getroot()
                real_mid = rt.findtext(".//measurement/id", "project")
                sample_node = rt.find(".//measurement/sample")
                if sample_node is not None:
                    sample_info = {
                        "Name":         sample_node.findtext("Name","BLANK"),
                        "Company":      sample_node.findtext("Company","BLANK"),
                        "Location":     sample_node.findtext("Location","BLANK"),
                        "WellID":       sample_node.findtext("WellID","BLANK"),
                        "CoreRun":      sample_node.findtext("CoreRun","0"),
                        "CoreBox":      sample_node.findtext("CoreBox","0"),
                        "Date":         sample_node.findtext("Date","BLANK"),
                        "Length":       sample_node.findtext("Length","0"),
                        "Diameter":     sample_node.findtext("Diameter","0"),
                        "Slabbed":      sample_node.findtext("Slabbed","(null)"),
                        "Geologist":    sample_node.findtext("Geologist","BLANK"),
                        "RecordedBy":   sample_node.findtext("RecordedBy","BLANK"),
                        "Comment":      sample_node.findtext("Comment",""),
                        "DepthInEarth": sample_node.findtext("DepthInEarth","0"),
                        "DepthInRock":  sample_node.findtext("DepthInRock","0"),
                        "TotalDepth From": sample_node.findtext("TotalDepth","0"),
                        "BottomElevation": sample_node.findtext("BottomElevation","0"),
                        "RockElevation":   sample_node.findtext("RockElevation","0"),
                        "CoreRecovery":    sample_node.findtext("CoreRecovery","100"),
                        "Measurement-ID":  real_mid
                    }
                depth_val = rt.findtext(".//Depth","NoDepth")
            except:
                sample_info= {}
            new_xml_name = f"{real_mid}.xml"
            final_xml_path = os.path.join(xml_dir, new_xml_name)
            shutil.move(xml_extracted, final_xml_path)

        # subfolder => {Depth}_{ancprjBase}_{measurementID}
        subfolder_name = f"{depth_val}_{base_noext}_{real_mid}"

        if final_xml_path and os.path.isfile(final_xml_path):
            create_hdrs_and_pad_optimized(final_xml_path, temp_folder, sample_info)

        swir_sub = os.path.join(swir_dir, subfolder_name)
        vnir_sub = os.path.join(vnir_dir, subfolder_name)
        os.makedirs(swir_sub, exist_ok=True)
        os.makedirs(vnir_sub, exist_ok=True)

        for fx in os.listdir(temp_folder):
            if not fx.lower().endswith(".hdr"):
                continue
            hdr_path = os.path.join(temp_folder, fx)
            raw_path = os.path.splitext(hdr_path)[0] + ".raw"
            if not os.path.isfile(raw_path):
                continue

            # parse camera
            cam = None
            with open_with_buffer(hdr_path, "r") as rr:
                lines = rr.readlines()
            for ll in lines:
                if ll.strip().startswith("CameraName"):
                    parts= ll.split("=",1)
                    if len(parts)==2:
                        cam= parts[1].strip()
                    break

            if cam and ("vnir" in cam.lower()):
                dst_hdr = os.path.join(vnir_sub, fx)
                dst_raw = os.path.join(vnir_sub, os.path.basename(raw_path))
                shutil.move(hdr_path, dst_hdr)
                shutil.move(raw_path, dst_raw)
                print(f"[SORTED: VNIR] => {fx} => {vnir_sub}")
                sys.stdout.flush()
            else:
                dst_hdr = os.path.join(swir_sub, fx)
                dst_raw = os.path.join(swir_sub, os.path.basename(raw_path))
                shutil.move(hdr_path, dst_hdr)
                shutil.move(raw_path, dst_raw)
                print(f"[SORTED: SWIR] => {fx} => {swir_sub}")
                sys.stdout.flush()

        shutil.rmtree(temp_folder)
        os.rename(zip_path, os.path.join(base_folder, f"{base_noext}_processed.ancprj"))
        print(f"[DONE] {zip_name} => {base_noext}_processed.ancprj")
        sys.stdout.flush()

    print("Extraction complete.\n")
    
def rename_and_extract():
    base_folder = folder_entry.get().strip()
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return
    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    try:
        extract_and_rename(base_folder)
        messagebox.showinfo("Success", "Extraction complete.")
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred during extraction:\n{e}")
    finally:
        sys.stdout = old_stdout
        

       
        
# CODE 2: REFLECTANCE CORRECTION

def load_envi_data(hdr_path, raw_path):
    """Load ENVI data from header and raw files"""
    with open(hdr_path, 'r') as f:
        contents = f.read()
    if "byte order" not in contents.lower():
        with open(hdr_path, 'a') as f:
            f.write("\nbyte order = 0\n")
    img = envi.open(hdr_path, raw_path)
    return img.load(), img.metadata

def get_output_filenames(input_hdr, output_folder):
    """Generate output filenames with _COR suffix"""
    base = os.path.splitext(os.path.basename(input_hdr))[0]
    new_base = base + "_COR"
    return os.path.join(output_folder, new_base + ".hdr"), os.path.join(output_folder, new_base + ".raw")

def load_panel_reflectance(panel_path):
    """Load panel reflectance data from CSV file"""
    print(f"Loading panel reflectance from: {panel_path}")
    
    try:
        # Read the file line by line
        with open(panel_path, 'r') as f:
            lines = f.readlines()
        
        if len(lines) < 3:
            raise ValueError("File has insufficient data lines")
        
        # Extract headers
        headers = lines[1].strip()
        
        if "nm" in headers and "%R" in headers:
            # Parse data lines
            wavelengths = []
            reflectances = []
            
            for line in lines[2:]:
                if not line.strip():
                    continue
                
                parts = line.strip().split(';')
                if len(parts) >= 2:
                    try:
                        wl = float(parts[0])
                        refl_str = parts[1].replace(',', '.')
                        refl = float(refl_str)
                        
                        wavelengths.append(wl)
                        reflectances.append(refl)
                    except ValueError:
                        continue
            
            if len(wavelengths) < 10:
                # Try alternative parsing if standard method fails
                for line in lines[2:]:
                    if not line.strip():
                        continue
                    for delimiter in [';', ',', ' ', '\t']:
                        parts = line.strip().split(delimiter)
                        if len(parts) >= 2:
                            try:
                                wl = float(parts[0])
                                refl_str = parts[1].replace(',', '.')
                                refl = float(refl_str)
                                wavelengths.append(wl)
                                reflectances.append(refl)
                                break
                            except:
                                continue
            
            if len(wavelengths) < 10:
                raise ValueError(f"Too few valid data points: {len(wavelengths)}")
            
            # Convert to numpy arrays
            nm = np.array(wavelengths, dtype=np.float32)
            ref_values = np.array(reflectances, dtype=np.float32)
            
            # Check if reflectance is in percentage
            if ref_values.max() > 1.5:
                ref = ref_values / 100
            else:
                ref = ref_values
            
            # Ensure reflectance values are between 0 and 1
            ref = np.clip(ref, 0, 1)
            
            print(f"Successfully loaded panel data: {len(nm)} points")
            return nm, ref
        else:
            raise ValueError("Expected column headers 'nm' and '%R' not found")
            
    except Exception as e:
        print(f"Error loading panel reflectance: {str(e)}")
        raise ValueError(f"Failed to parse panel data: {str(e)}")

def calibrate_white_reference(measured_white, white_meta, panel_nm, panel_ref):
    """Calibrate white reference using panel reflectance data"""
    w = measured_white.astype(np.float32)
    if "wavelength" not in white_meta:
        raise ValueError("No 'wavelength' in WR header.")
    band_centers = np.array([float(x) for x in white_meta["wavelength"]], dtype=np.float32)
    panel_vals = np.interp(band_centers, panel_nm, panel_ref).reshape((1, 1, -1))
    panel_vals[panel_vals < 1e-6] = 1e-6  # Avoid division by zero
    return w / panel_vals

def correct_raw_hsi_with_calibrated_wh(calib_white, raw_data):
    """Correct raw HSI data using calibrated white reference"""
    cwhite = calib_white.copy()
    cwhite[cwhite < 1e-6] = 1e-6  # Avoid division by zero
    raw_f32 = raw_data.astype(np.float32)
    return raw_f32 / cwhite

def update_metadata(meta):
    """Update metadata for corrected files"""
    meta["data type"] = 4  # float32
    meta["byte order"] = 0
    meta["interleave"] = "bil"
    meta["file type"] = "ENVI"
    return meta

def get_white_reference_for_hsi(hsi_folder, white_ref_dir, sensor_type):
    """Find matching white reference file for an HSI folder"""
    folder_name = os.path.basename(hsi_folder)
    m = re.match(r'^[^_]+_([^_]+(?:_[^_]+)?)_\{', folder_name)
    if not m:
        print(f"Could not extract token from folder: {folder_name}")
        return None, None, None
    
    token = m.group(1)
    candidates = []
    
    for file in os.listdir(white_ref_dir):
        if file.lower().endswith('.hdr') and sensor_type.lower() in file.lower():
            if token.lower() in file.lower() and "neoreftarget" in file.lower():
                wr_path = os.path.join(white_ref_dir, file)
                base = os.path.splitext(file)[0]
                raw_candidate = os.path.join(white_ref_dir, base + ".raw")
                if not os.path.exists(raw_candidate):
                    raw_candidate = os.path.join(white_ref_dir, base + ".envi")
                if os.path.exists(raw_candidate):
                    candidates.append((wr_path, raw_candidate))
    
    if candidates:
        wr_hdr, wr_raw = candidates[0]
        print(f"Using {sensor_type} WR with token '{token}'")
        return wr_hdr, wr_raw, token
    else:
        print(f"No matching {sensor_type} white reference for token '{token}'")
        return None, None, None

def get_reference_panel_definition(white_ref_dir, token, sensor_type):
    """Find matching reference panel definition file"""
    for file in os.listdir(white_ref_dir):
        if file.lower().endswith('.csv') and "referencepaneldefinition" in file.lower():
            if token.lower() in file.lower() and sensor_type.lower() in file.lower():
                panel_path = os.path.join(white_ref_dir, file)
                print(f"Using Reference Panel Definition for {sensor_type} with token '{token}'")
                return panel_path
    
    print(f"No matching Reference Panel Definition for {sensor_type} with token '{token}'")
    return None

def process_directory_with_calib(inp, outp, wr_dir, sensor_type):
    """Process a directory of HSI files with calibrated reflectance correction"""
    used_files = set()
    
    for root, _, files in os.walk(inp):
        if not any(f.lower().endswith('.hdr') for f in files):
            continue
        
        # Get white reference file and token (uses actual sensor_type)
        wr_hdr, wr_raw, token = get_white_reference_for_hsi(root, wr_dir, sensor_type)
        if not (wr_hdr and wr_raw):
            print(f"Skipping folder due to missing white reference: {root}")
            continue
        
        # Get panel definition file (ALWAYS uses "SWIR" regardless of actual sensor_type)
        panel_xlsx = get_reference_panel_definition(wr_dir, token, "SWIR")
        if not panel_xlsx:
            print(f"Skipping folder due to missing reference panel definition: {root}")
            continue
        
        temp_files = {wr_hdr, wr_raw, panel_xlsx}
        
        try:
            # Load white reference data
            wdata, wmeta = load_envi_data(wr_hdr, wr_raw)
            wdata = wdata.mean(axis=0, keepdims=True)
            
            # Load panel reflectance data
            panel_nm, panel_ref = load_panel_reflectance(panel_xlsx)
            
            # Calibrate white reference
            calib_white = calibrate_white_reference(wdata, wmeta, panel_nm, panel_ref)
            
            # Create output directory
            rel_path = os.path.relpath(root, inp)
            tgt_dir = os.path.join(outp, rel_path)
            os.makedirs(tgt_dir, exist_ok=True)
            
            files_processed = False
            
            # Process each HSI file
            for f in files:
                if f.lower().endswith('.hdr') and '_cor' not in f.lower():
                    hdr_path = os.path.join(root, f)
                    base = os.path.splitext(f)[0]
                    raw_path = os.path.join(root, base + ".raw")
                    
                    if not os.path.exists(raw_path):
                        continue
                    
                    try:
                        # Load raw HSI data
                        raw_data, raw_meta = load_envi_data(hdr_path, raw_path)
                        
                        # Apply reflectance correction
                        refl = correct_raw_hsi_with_calibrated_wh(calib_white, raw_data)
                        
                        # Update metadata and save
                        new_meta = update_metadata(raw_meta)
                        out_hdr, out_raw = get_output_filenames(hdr_path, tgt_dir)
                        os.makedirs(os.path.dirname(out_hdr), exist_ok=True)
                        envi.save_image(out_hdr, refl, force=True,
                                      interleave=new_meta["interleave"],
                                      metadata=new_meta, dtype=np.float32)
                        
                        print(f"Processed: {os.path.basename(hdr_path)}")
                        
                        # Remove original files
                        os.remove(hdr_path)
                        os.remove(raw_path)
                        files_processed = True
                    
                    except Exception as e:
                        print(f"Error processing {os.path.basename(hdr_path)}: {e}")
            
            if files_processed:
                used_files.update(temp_files)
                
        except Exception as e:
            print(f"Error processing folder: {e}")
    
    return used_files

def cleanup_empty_folders(directory):
    """Remove empty directories after processing"""
    for root, dirs, _ in os.walk(directory, topdown=False):
        for d in dirs:
            dp = os.path.join(root, d)
            if not os.listdir(dp):
                os.rmdir(dp)
                print(f"Removed empty directory: {os.path.basename(dp)}")

def delete_files(paths):
    """Delete files with error handling"""
    for p in paths:
        try:
            os.remove(p)
            print(f"Deleted used file: {os.path.basename(p)}")
        except Exception as e:
            print(f"Failed to delete {os.path.basename(p)}: {e}")

def reflect_correct_hsi():
    """Main function to perform reflectance correction"""
    base_folder = folder_entry.get().strip()
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return
    
    swir_input = os.path.join(base_folder, "SWIR_HSI_Files")
    vnir_input = os.path.join(base_folder, "VNIR_HSI_Files")
    white_ref_dir = os.path.join(base_folder, "Extracted_WR")
    
    # Check required folders exist
    missing = []
    for d in [swir_input, vnir_input, white_ref_dir]:
        if not os.path.isdir(d):
            missing.append(os.path.basename(d))
    if missing:
        messagebox.showerror("Error", f"Missing required folders: {', '.join(missing)}")
        return
    
    # Create output directories
    corrected_swir = os.path.join(base_folder, "Corrected_SWIR")
    corrected_vnir = os.path.join(base_folder, "Corrected_VNIR")
    os.makedirs(corrected_swir, exist_ok=True)
    os.makedirs(corrected_vnir, exist_ok=True)
    
    # Redirect stdout to log widget
    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    
    try:
        # Process SWIR data
        print("Starting Reflectance Correction for SWIR...")
        used_swir_files = process_directory_with_calib(swir_input, corrected_swir, white_ref_dir, "SWIR")
        cleanup_empty_folders(swir_input)
        print("Completed SWIR reflectance correction.")
        
        # Process VNIR data
        print("Starting Reflectance Correction for VNIR...")
        used_vnir_files = process_directory_with_calib(vnir_input, corrected_vnir, white_ref_dir, "VNIR")
        cleanup_empty_folders(vnir_input)
        print("Completed VNIR reflectance correction.")
        
        # Delete used files
        delete_files(used_swir_files)
        delete_files(used_vnir_files)
        
        messagebox.showinfo("Success", "Reflectance correction completed successfully.")
    
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred during reflectance correction:\n{e}")
    
    finally:
        sys.stdout = old_stdout




# CODE 3: Coregister SWIR-VNIR corrected files 

def coregister_hsi_optimized():
    """Optimized version of the coregistration function"""
    base_folder = folder_entry.get().strip()
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return
    
    swir_dir = os.path.join(base_folder, "Corrected_SWIR")
    vnir_dir = os.path.join(base_folder, "Corrected_VNIR")
    output_dir = os.path.join(base_folder, "Coregistered_HSI")
    
    # Check if required directories exist
    if not os.path.isdir(swir_dir) or not os.path.isdir(vnir_dir):
        messagebox.showerror("Error", "Corrected_SWIR and/or Corrected_VNIR folders are missing.")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Hardcoded fixed location of transform parameters file
    transform_file = TRANSFORM_PARAMETERS_FILE
    
    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    
    try:
        print("Starting VNIR/SWIR coregistration...")
        sys.stdout.flush()
        
        # Load transform matrix from file 
        def load_affine(txt_file):
            try:
                with open(txt_file, 'r') as f:
                    rows = [l for l in f if l.strip() and not l.startswith("#")]
                    mat = np.array([[float(x) for x in r.split()[:3]] for r in rows[:3]])
                    return mat
            except Exception as e:
                raise ValueError(f"Failed to load transform parameters: {e}")
        
        # Load transform matrix
        try:
            transform_matrix = load_affine(transform_file)
            print(f"Using transform matrix:\n{transform_matrix}")
            sys.stdout.flush()
        except Exception as e:
            messagebox.showerror("Error", str(e))
            return
            
        # helper functions 
        def remove_duplicate_lines_fast(cube, max_top_check=30, similarity=0.999):
            for i in range(min(max_top_check, cube.shape[0] - 1)):
                a, b = cube[i].ravel(), cube[i + 1].ravel()
                if (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9) < similarity:
                    return i
            return 0

        def resample_image(cube, new_h, new_w):
            z_y, z_x = new_h / cube.shape[0], new_w / cube.shape[1]
            return zoom(cube, (z_y, z_x, 1), order=1)

        def get_wavelengths(meta):
            return np.asarray([float(w) for w in meta.get("wavelength", [])], float)
        
        # Version-proof affine warp 
        _warp_sig = inspect.signature(warp).parameters
        if "channel_axis" in _warp_sig:
            def _warp_cube(cube, tform):
                return warp(cube, tform.inverse, order=1, mode="edge",
                          preserve_range=True, channel_axis=-1).astype(cube.dtype)
        elif "multichannel" in _warp_sig:
            def _warp_cube(cube, tform):
                return warp(cube, tform.inverse, order=1, mode="edge",
                          preserve_range=True, multichannel=True).astype(cube.dtype)
        else:
            def _warp_cube(cube, tform):
                out = np.empty_like(cube)
                for b in range(cube.shape[2]):
                    out[:, :, b] = warp(cube[:, :, b], tform.inverse,
                                      order=1, mode="edge",
                                      preserve_range=True)
                return out
        
        # Fast composite function 
        def create_cross_calibrated_composite_fast(vnir, swir, vnir_wv, swir_wv,
                                                transition_width=5.0):
            vnir, swir = vnir.astype(np.float32), swir.astype(np.float32)
            vnir_wv, swir_wv = map(np.asarray, (vnir_wv, swir_wv))
            ov_min, ov_max = max(vnir_wv[0], swir_wv[0]), min(vnir_wv[-1], swir_wv[-1])
            tr_c = ov_max - transition_width
            tr_s, tr_e = tr_c - transition_width / 2, tr_c + transition_width / 2

            v_only = np.where(vnir_wv < tr_s)[0]
            s_only = np.where(swir_wv > tr_e)[0]
            v_cal  = np.where((vnir_wv >= ov_min) & (vnir_wv <= ov_max))[0]
            s_cal  = np.where((swir_wv >= ov_min) & (swir_wv <= ov_max))[0]

            min_len = min(len(v_cal), len(s_cal))
            if min_len == 0:                       
                scale = np.ones(vnir.shape[:2], np.float32)
            else:
                ratio = np.divide(
                    swir[:, :, s_cal[:min_len]],      
                    vnir[:, :, v_cal[:min_len]],      
                    out=np.full_like(vnir[:, :, v_cal[:min_len]], np.nan),
                    where=vnir[:, :, v_cal[:min_len]] > 1e-2
                )
                scale = np.nanmedian(ratio, axis=-1)
                scale[np.isnan(scale)] = 1.0
            v_scaled = vnir * scale[..., None]

            L, S, _ = swir.shape
            trans_wv = np.linspace(tr_s, tr_e, 5)
            comp = np.empty((L, S, len(v_only) + len(trans_wv) + len(s_only)),
                          dtype=np.float32)

            comp[:, :, :len(v_only)] = v_scaled[:, :, v_only]

            wts = 3*(np.linspace(0,1,5)**2) - 2*(np.linspace(0,1,5)**3)
            for k, w in enumerate(wts):
                i_out = len(v_only)+k
                i_v   = vnir_wv.searchsorted(trans_wv[k])
                i_s   = swir_wv.searchsorted(trans_wv[k])
                comp[:, :, i_out] = (1-w)*v_scaled[:, :, i_v] + w*swir[:, :, i_s]

            comp[:, :, len(v_only)+len(trans_wv):] = swir[:, :, s_only]
            out_wv = np.concatenate([vnir_wv[v_only], trans_wv, swir_wv[s_only]])
            return comp, out_wv.astype(str).tolist()

        # Optimized process_pair function
        def process_pair(swir_hdr, vnir_hdr, out_dir, out_name, tform):
            # Skip if input files are already marked as processed (have _crg suffix)
            if "_crg" in swir_hdr or "_crg" in vnir_hdr:
                return False, "skipped_processed"
                
            # Check if output file already exists
            out_path = os.path.join(out_dir, out_name + ".hdr")
            if os.path.exists(out_path):
                return False, "skipped_exists"
                
            os.makedirs(out_dir, exist_ok=True)
        
            swir_img = envi.open(swir_hdr)
            vnir_img = envi.open(vnir_hdr)
            
            # Load data
            swir = swir_img.load().astype(np.float32)
            vnir = vnir_img.load().astype(np.float32)

            # Process SWIR image
            swir = np.flip(swir, axis=1)
            dup = remove_duplicate_lines_fast(swir)
            if dup > 0:
                swir = swir[dup:]

            # Resample VNIR if needed
            if swir.shape[:2] != vnir.shape[:2]:
                vnir = resample_image(vnir, *swir.shape[:2])

            # Apply transformation
            vnir = _warp_cube(vnir, tform)

            # Create composite
            comp, comp_wv = create_cross_calibrated_composite_fast(
                vnir, swir, 
                get_wavelengths(vnir_img.metadata), 
                get_wavelengths(swir_img.metadata)
            )

            # Update metadata
            meta = swir_img.metadata.copy()
            meta.update(
                bands=len(comp_wv), 
                wavelength=comp_wv,
                **{
                    "acquisition date": datetime.now().strftime("%Y-%m-%d"),
                    "acquisition time": datetime.now().strftime("%H:%M:%S")
                }
            )

            # Save composite (without _crg suffix in output)
            envi.save_image(
                out_path, 
                comp, 
                force=True, 
                dtype=np.float32,
                interleave=meta.get("interleave", "bil"),
                metadata=meta
            )
            
            # Mark the input files as processed by renaming them with _crg suffix
            try:
                # Rename SWIR files (.hdr and corresponding .img)
                swir_base, swir_ext = os.path.splitext(swir_hdr)
                swir_img = swir_base + ".img"
                
                new_swir_hdr = f"{swir_base}_crg{swir_ext}"
                new_swir_img = f"{swir_base}_crg.img"
                
                os.rename(swir_hdr, new_swir_hdr)
                if os.path.exists(swir_img):
                    os.rename(swir_img, new_swir_img)
                
                # Rename VNIR files (.hdr and corresponding .img)
                vnir_base, vnir_ext = os.path.splitext(vnir_hdr)
                vnir_img = vnir_base + ".img"
                
                new_vnir_hdr = f"{vnir_base}_crg{vnir_ext}"
                new_vnir_img = f"{vnir_base}_crg.img"
                
                os.rename(vnir_hdr, new_vnir_hdr)
                if os.path.exists(vnir_img):
                    os.rename(vnir_img, new_vnir_img)
            except Exception as e:
                print(f" - Warning: Could not rename input files: {e}")
            
            return True, "processed"

        # Folder and filename matching from original code
        _depth = re.compile(r'(\d+\.\d+_\d+\.\d+)')
        _guid  = re.compile(r'{([^}]+)}')

        def folder_pairs(root_a, root_b):
            a = {d.split("{")[0]: d for d in os.listdir(root_a) if "{" in d}
            b = {d.split("{")[0]: d for d in os.listdir(root_b) if "{" in d}
            pairs = []
            for k in a.keys() & b.keys():
                guid_a = a[k][a[k].find("{")+1:a[k].find("}")]
                guid_b = b[k][b[k].find("{")+1:b[k].find("}")]
                out_subdir = f"{k}{{{guid_b}}}_{{{guid_a}}}"
                pairs.append((
                    os.path.join(root_a, a[k]),
                    os.path.join(root_b, b[k]),
                    out_subdir
                ))
            return pairs

        def hdr_pairs(dir_a, dir_b):
            # Build lookup dictionary just once
            a_files = glob.glob(os.path.join(dir_a, "*.hdr"))
            A = {}
            for f in a_files:
                m = _depth.search(f)
                if m:
                    A[m.group(1)] = f
            
            # Find matching files
            pairs = []
            for vn in glob.glob(os.path.join(dir_b, "*.hdr")):
                d = _depth.search(vn)
                if not d or d.group(1) not in A:
                    continue
                    
                sw = A[d.group(1)]
                m_sw = _guid.search(sw)
                m_vn = _guid.search(vn)
                
                gid_sw = m_sw.group(1) if m_sw else "unknown"
                gid_vn = m_vn.group(1) if m_vn else "unknown"
                
                name = f"{d.group(1)}_{{{gid_vn}}}_{{{gid_sw}}}"
                pairs.append((sw, vn, name))
            
            return pairs

        # Main processing with optimization
        tform = AffineTransform(matrix=transform_matrix)
        t0 = time.time()
        
        # Use batch reporting instead of per-file
        total = processed = skipped = failed = 0
        skipped_processed = skipped_exists = 0
        
        # Get all folder pairs upfront (avoid generator)
        all_folder_pairs = folder_pairs(swir_dir, vnir_dir)
        
        # Process each folder pair
        for idx, (sw_dir, vn_dir, out_sub) in enumerate(all_folder_pairs):
            folder_name = os.path.basename(sw_dir)
            print(f"Processing folder pair {idx+1}/{len(all_folder_pairs)}: {folder_name}")
            sys.stdout.flush()
            
            # Get all file pairs upfront
            file_pairs = hdr_pairs(sw_dir, vn_dir)
            processed_in_folder = 0
            
            # Process all files in the folder
            for sw_hdr, vn_hdr, name in file_pairs:
                total += 1
                try:
                    result, status = process_pair(sw_hdr, vn_hdr, 
                                      os.path.join(output_dir, out_sub), name, tform)
                    if result:
                        processed += 1
                        processed_in_folder += 1
                    else:
                        skipped += 1
                        if status == "skipped_processed":
                            skipped_processed += 1
                        elif status == "skipped_exists":
                            skipped_exists += 1
                except Exception as e:
                    print(f" - Error processing {os.path.basename(sw_hdr)}: {e}")
                    failed += 1
            
            # Report only after each folder is complete
            if processed_in_folder > 0:
                print(f" - Processed {processed_in_folder} files")
            elif skipped > 0:
                print(f" - All files already processed")
            sys.stdout.flush()
        
        # Final report
        elapsed = time.time() - t0
        print(f"\nFinished coregistration:")
        print(f" - {processed} files processed")
        print(f" - {skipped_processed} files skipped (already processed)")
        print(f" - {skipped_exists} files skipped (output exists)")
        print(f" - {failed} files failed")
        print(f"Total time: {elapsed:.1f}s")
        sys.stdout.flush()
        
        messagebox.showinfo("Success", f"Coregistration completed: {processed} files processed")
    
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred during coregistration:\n{e}")
    
    finally:
        sys.stdout = old_stdout




# CODE 4: Mask HSI Files

def mask_hsi_files():
    """Apply masks to HSI files using a pre-trained model via subprocess"""
    import tempfile
    import subprocess
    
    # Get input paths
    mask_path = mask_entry.get().strip()
    base_folder = folder_entry.get().strip()
    
    # Check for valid input paths
    if not mask_path:
        # If mask_path not provided, check for Coregistered_HSI in base_folder
        mask_path = os.path.join(base_folder, "Coregistered_HSI")
    
    if not os.path.isdir(mask_path):
        messagebox.showerror("Error", "Files to be masked directory not found. Please provide a valid path.")
        return
    
    # Hardcoded model path
    model_path = MASK_MODEL_PATH
    if not os.path.isfile(model_path):
        messagebox.showerror("Error", f"Model file not found at: {model_path}")
        return
    
    # Define output directory based on which input was used
    if not mask_entry.get().strip():  # Using fallback location
        # Using the default fallback path (Coregistered_HSI in main folder)
        output_dir = os.path.join(base_folder, "Masked_Coreg_HSI")
    else:
        # Using the specific mask_entry path
        output_dir = os.path.join(os.path.dirname(mask_path), "Masked_HSI")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Create a temporary Python script file that will run the masking process
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as script_file:
        script_path = script_file.name
        script_file.write('''
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import cv2
import segmentation_models_pytorch as smp
from PIL import Image
import re

ENCODER = 'efficientnet-b4'  
ENCODER_WEIGHTS = 'imagenet'  

def clean_directory_name(name):
    """Clean name by replacing first UUID with VNIR, keep second UUID"""
    # Pattern to match UUIDs like {185CFCDE-351D-40B4-A01A-F161F535B46C}
    uuid_pattern = r'\\{[A-F0-9]{8}-[A-F0-9]{4}-[A-F0-9]{4}-[A-F0-9]{4}-[A-F0-9]{12}\\}'
    
    # Replace only the first UUID with VNIR, keep the second UUID
    cleaned = re.sub(uuid_pattern, '{VNIR}', name, count=1)
    
    return cleaned

def read_hdr(hdr_file):
    hdr = {}
    with open(hdr_file, 'r') as f:
        for line in f:
            if '=' in line:
                key, value = line.split('=', 1)
                hdr[key.strip().lower()] = value.strip()
    return hdr

def normalize_band(band):
    norm = np.clip(band, 0, 1)
    return (norm * 255).astype(np.uint8)

def pad_to_divisible_by_32(image):
    """Pad image to make dimensions divisible by 32"""
    height, width = image.shape[:2]
    
    # Calculate required padding
    pad_height = (32 - height % 32) % 32
    pad_width = (32 - width % 32) % 32
    
    # Pad the image (symmetric padding)
    if pad_height > 0 or pad_width > 0:
        padded_image = cv2.copyMakeBorder(
            image,
            top=0,
            bottom=pad_height,
            left=0,
            right=pad_width,
            borderType=cv2.BORDER_REFLECT
        )
        return padded_image, (height, width), (pad_height, pad_width)
    
    return image, (height, width), (0, 0)

def preprocess_image(image, preprocessing_fn):
    """Preprocess image for inference with padding"""
    # Store original dimensions
    original_shape = image.shape[:2]
    
    # Pad image to be divisible by 32
    image, original_dims, padding = pad_to_divisible_by_32(image)
    
    # Apply preprocessing (normalization)
    image = preprocessing_fn(image)
    
    # Convert to tensor and add batch dimension
    image_tensor = torch.from_numpy(image.transpose(2, 0, 1)).float().unsqueeze(0)
    
    return image_tensor, original_dims, padding

def process_hsi_files(input_dir, output_dir, model_path, px_to_reduce=2):
    """Main function to mask HSI files"""
    # Normalize all paths
    input_dir = os.path.normpath(input_dir)
    output_dir = os.path.normpath(output_dir)
    model_path = os.path.normpath(model_path)
    
    print(f"Input directory: {input_dir}")
    print(f"Output directory: {output_dir}")
    print(f"Model path: {model_path}")
    
    # Setup device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Create output directories
    rgb_output_dir = os.path.join(output_dir, "false_RGB")
    mask_output_dir = os.path.join(output_dir, "false_RGB_Masks")
    os.makedirs(rgb_output_dir, exist_ok=True)
    os.makedirs(mask_output_dir, exist_ok=True)
    
    # Load model
    try:
        model = torch.load(model_path, map_location=device, weights_only=False)
        model = model.to(device)
        model.eval()
        print(f"Successfully loaded model from {model_path}")
        
        # Get preprocessing function for the model
        preprocessing_fn = smp.encoders.get_preprocessing_fn(ENCODER, ENCODER_WEIGHTS)
        print(f"Using encoder: {ENCODER} with weights: {ENCODER_WEIGHTS}")
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return False
    
    # Setup bands for RGB visualization
    bands_idx = [50, 100, 150]  # Default RGB visualization bands
    
    # Find all .img files
    img_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith('.img') and '_mskd' not in file:
                img_files.append(os.path.join(root, file))
    
    print(f"Found {len(img_files)} HSI files to process")
    
    processed_count = 0
    skipped_count = 0
    
    for img_file in img_files:
        try:
            # Get original file info
            file_basename = os.path.basename(img_file)
            rel_path = os.path.relpath(os.path.dirname(img_file), input_dir)
            rel_path = os.path.normpath(rel_path)
            
            # Clean names by replacing UUIDs
            rel_path_clean = clean_directory_name(rel_path)
            file_basename_clean = clean_directory_name(file_basename)
            
            print(f"Processing: {file_basename}")
            print(f"  Directory: {rel_path} → {rel_path_clean}")
            print(f"  Filename: {file_basename} → {file_basename_clean}")
            
            hdr_file = os.path.splitext(img_file)[0] + '.hdr'
            if not os.path.exists(hdr_file):
                print(f"  Missing HDR file, skipping")
                continue
            
            # Check if already processed
            if '_mskd' in file_basename:
                print(f"  Already processed, skipping")
                skipped_count += 1
                continue
                
            # Check if output already exists
            output_img_path = os.path.join(output_dir, rel_path_clean, file_basename_clean)
            if os.path.exists(output_img_path):
                print(f"  Output already exists, skipping")
                skipped_count += 1
                continue
            
            # Read header file
            hdr = read_hdr(hdr_file)
            samples = int(hdr.get('samples', 0))
            lines = int(hdr.get('lines', 0))
            bands = int(hdr.get('bands', 0))
            
            if samples == 0 or lines == 0 or bands == 0:
                print(f"  Invalid header information, skipping")
                continue
            
            # Determine data type
            dtype_str = hdr.get('data type', '4')
            if dtype_str == '4':
                dtype = np.float32
            elif dtype_str == '5':
                dtype = np.float64
            elif dtype_str == '12':
                dtype = np.uint16
            else:
                dtype = np.float32
            
            # Read raw data
            with open(img_file, 'rb') as f:
                raw_data = np.fromfile(f, dtype=dtype)
            
            if raw_data.size != samples * lines * bands:
                print(f"  Size mismatch, skipping")
                continue
            
            # Reshape according to interleave format
            interleave = hdr.get('interleave', 'bil').lower()
            if interleave == 'bil':
                raw_data = raw_data.reshape((lines, bands, samples))
                raw_data = raw_data.transpose(0, 2, 1)  # to [lines, samples, bands]
            elif interleave == 'bip':
                raw_data = raw_data.reshape((lines, samples, bands))
            elif interleave == 'bsq':
                raw_data = raw_data.reshape((bands, lines, samples))
                raw_data = raw_data.transpose(1, 2, 0)  # to [lines, samples, bands]
            
            # Create false color RGB image
            band_indices = [min(idx, bands-1) for idx in bands_idx]
            r, g, b = [normalize_band(raw_data[:, :, idx]) for idx in band_indices]
            rgb = np.stack([r, g, b], axis=-1)
            
            # Apply initial edge masking
            rgb[:, :px_to_reduce, :] = 0
            rgb[:, -px_to_reduce:, :] = 0
            
            # Create RGB output directory
            output_rgb_subfolder = os.path.join(rgb_output_dir, rel_path_clean)
            os.makedirs(output_rgb_subfolder, exist_ok=True)
            
            # Save RGB image
            rgb_output_path = os.path.join(output_rgb_subfolder, f"{os.path.splitext(file_basename_clean)[0]}.png")
            plt.imsave(rgb_output_path, rgb)
            print(f"  Saved RGB image")
            
            # Generate mask using the new model approach
            # Preprocess image with padding
            image_tensor, original_dims, padding = preprocess_image(rgb, preprocessing_fn)
            
            # Move to device
            image_tensor = image_tensor.to(device)
            
            # Get prediction
            with torch.no_grad():
                output = model(image_tensor)
                
                # Convert to numpy and squeeze dimensions
                mask = output.squeeze().cpu().numpy()
                
                # Apply threshold to get binary mask
                binary_mask = (mask > 0.5).astype(np.uint8) * 255
                
                # Remove padding from the mask to get back to original dimensions
                if padding[0] > 0 or padding[1] > 0:
                    binary_mask = binary_mask[:original_dims[0], :original_dims[1]]
                
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
                binary_mask = cv2.erode(binary_mask, kernel, iterations=1)
                
                # Post-processing: Remove white elements that don't have at least one pixel at width/2 position
                center_col = binary_mask.shape[1] // 2
                num_labels, labels = cv2.connectedComponents(binary_mask)
                for label in range(1, num_labels):
                    component_mask = (labels == label)
                    if not np.any(component_mask[:, center_col]):
                        binary_mask[component_mask] = 0
                
                # Fill holes inside white regions
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.fillPoly(binary_mask, contours, 255)
                
                # Convert back to 0-1 for masking
                binary_mask = (binary_mask > 0).astype(np.uint8)
            
            print(f"  Generated mask with new model")
            
            # Create mask output directory
            mask_output_subfolder = os.path.join(mask_output_dir, rel_path_clean)
            os.makedirs(mask_output_subfolder, exist_ok=True)
            
            # Save mask image
            mask_output_path = os.path.join(mask_output_subfolder, f"{os.path.splitext(file_basename_clean)[0]}.png")
            mask_image = Image.fromarray(binary_mask * 255)
            mask_image.save(mask_output_path)
            print(f"  Saved mask image")
            
            # Apply mask to raw data
            mask_resized = np.repeat(binary_mask[:, :, np.newaxis], raw_data.shape[2], axis=2)
            raw_data[mask_resized == 0] = 0
            
            # Apply edge masking to raw data
            raw_data[:, :px_to_reduce, :] = 0
            raw_data[:, -px_to_reduce:, :] = 0
            
            # Create output directory for final files
            output_subfolder = os.path.join(output_dir, rel_path_clean)
            os.makedirs(output_subfolder, exist_ok=True)
            
            # Save masked HSI data
            output_img_path = os.path.join(output_subfolder, file_basename_clean)
            output_hdr_path = os.path.join(output_subfolder, os.path.splitext(file_basename_clean)[0] + '.hdr')
            
            # Transpose back to original format
            if interleave == 'bil':
                raw_to_save = raw_data.transpose(0, 2, 1)  # back to [lines, bands, samples]
            elif interleave == 'bsq':
                raw_to_save = raw_data.transpose(2, 0, 1)  # back to [bands, lines, samples]
            else:  # bip
                raw_to_save = raw_data
            
            raw_to_save = np.ascontiguousarray(raw_to_save)
            with open(output_img_path, 'wb') as f:
                raw_to_save.tofile(f)
            
            # Copy header file
            with open(hdr_file, 'r') as src, open(output_hdr_path, 'w') as dst:
                for line in src:
                    dst.write(line)
            
            print(f"  Saved masked HSI data")
            
            # Generate masked RGB preview
            r_masked, g_masked, b_masked = [normalize_band(raw_data[:, :, idx]) for idx in band_indices]
            rgb_masked = np.stack([r_masked, g_masked, b_masked], axis=-1)
            
            masked_rgb_path = os.path.join(output_rgb_subfolder, f"{os.path.splitext(file_basename_clean)[0]}_masked.png")
            plt.imsave(masked_rgb_path, rgb_masked)
            print(f"  Saved masked RGB preview")
            
            # Rename original files with _mskd suffix
            input_base_no_ext = os.path.splitext(img_file)[0]
            new_img_path = f"{input_base_no_ext}_mskd.img"
            new_hdr_path = f"{input_base_no_ext}_mskd.hdr"
            
            try:
                os.rename(img_file, new_img_path)
                os.rename(hdr_file, new_hdr_path)
                print(f"  Renamed original files with _mskd suffix")
            except Exception as e:
                print(f"  Warning: Could not rename original files: {e}")
            
            processed_count += 1
            print(f"  ✓ Successfully processed")
            
        except Exception as e:
            print(f"Error processing {file_basename}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print(f"\\nMasking process completed:")
    print(f"  - {processed_count} files processed")
    print(f"  - {skipped_count} files skipped")
    
    return processed_count > 0

if __name__ == "__main__":
    input_dir = sys.argv[1]
    output_dir = sys.argv[2]
    model_path = sys.argv[3]
    
    process_hsi_files(input_dir, output_dir, model_path)
''')
    
    # Set up device and model
    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    
    try:
        print("Starting HSI masking process using subprocess...")
        print(f"Input directory: {mask_path}")
        print(f"Output directory: {output_dir}")
        print(f"Model path: {model_path}")
        sys.stdout.flush()
        
        # Run the subprocess with proper environment variables
        env = os.environ.copy()
        env["PYTORCH_ALLOW_UNSAFE_SERIALIZATION"] = "1"
        
        process = subprocess.Popen(
            [sys.executable, script_path, mask_path, output_dir, model_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            env=env
        )
        
        # Stream the output from the subprocess
        for line in process.stdout:
            print(line.strip())
            sys.stdout.flush()
        
        # Wait for the process to complete
        return_code = process.wait()
        
        if return_code == 0:
            messagebox.showinfo("Success", "HSI masking completed successfully!")
        else:
            messagebox.showerror("Error", f"HSI masking failed with return code {return_code}")
    
    except Exception as e:
        error_msg = f"An error occurred during masking:\n{str(e)}"
        print(error_msg)
        sys.stdout.flush()
        messagebox.showerror("Error", error_msg)
    
    finally:
        sys.stdout = old_stdout
        # Clean up the temporary script file
        try:
            os.unlink(script_path)
        except:
            pass




#  CODE 5: Process XRF  

SHORT_LABEL_MAP = {
    "Zr Ka": "Zirconium - b'K' b'a'",
    "Zn Kb": "Zinc - b'K' b'b'",
    "Zn Ka": "Zinc - b'K' b'a'",
    "Y Kb": "Yttrium - b'K' b'b'",
    "Y Ka": "Yttrium - b'K' b'a'",
    "Xe Lb": "Xenon - b'L' b'b'",
    "Xe La": "Xenon - b'L' b'a'",
    "W Lb": "Tungsten - b'L' b'b'",
    "W La": "Tungsten - b'L' b'a'",
    "V Kb": "Vanadium - b'K' b'b'",
    "V Ka": "Vanadium - b'K' b'a'",
    "Tl Lb": "Thallium - b'L' b'b'",
    "Ti Kb": "Titanum - b'K' b'b'",
    "Ti Ka": "Titanum - b'K' b'a'",
    "Te Ka": "Tellurium - b'K' b'a'",
    "Ta Lb": "Tantalum - b'L' b'b'",
    "Ta La": "Tantalum - b'L' b'a'",
    "Sr Kb": "Strontium - b'K' b'b'",
    "Sr Ka": "Strontium - b'K' b'a'",
    "Sn Kb": "Tin - b'K' b'b'",
    "Sn Ka": "Tin - b'K' b'a'",
    "Si Ka": "Silicon - b'K' b'a'",
    "Se Kb": "Selenium - b'K' b'b'",
    "Se Ka": "Selenium - b'K' b'a'",
    "Sc Kb": "Scandium - b'K' b'b'",
    "Sc Ka": "Scandium - b'K' b'a'",
    "Sb Kb": "Antimony - b'K' b'b'",
    "Sb Ka": "Antimony - b'K' b'a'",
    "S Kb": "Sulfur - b'K' b'b'",
    "S Ka": "Sulfur - b'K' b'a'",
    "Rh Kb": "Rhodium - b'K' b'b'",
    "Rh Ka": "Rhodium - b'K' b'a'",
    "Re Lb": "Rhenium - b'L' b'b'",
    "Re La": "Rhenium - b'L' b'a'",
    "Pt Lb": "Platinum - b'L' b'b'",
    "Pt La": "Platinum - b'L' b'a'",
    "Pd Kb": "Palladium - b'K' b'b'",
    "Pd Ka": "Palladium - b'K' b'a'",
    "Pb Lb": "Lead - b'L' b'b'",
    "Pb La": "Lead - b'L' b'a'",
    "Os Lb": "Osmium - b'L' b'b'",
    "Os La": "Osmium - b'L' b'a'",
    "Ni Kb": "Nickel - b'K' b'b'",
    "Ni Ka": "Nickel - b'K' b'a'",
    "Nb Kb": "Niobium - b'K' b'b'",
    "Nb Ka": "Niobium - b'K' b'a'",
    "Mo Kb": "Molybdenium - b'K' b'b'",
    "Mo Ka": "Molybdenium - b'K' b'a'",
    "Mn Kb": "Manganese - b'K' b'b'",
    "Mn Ka": "Manganese - b'K' b'a'",
    "Mg Ka": "Magnesium - b'K' b'a'",
    "Kr Kb": "Krypton - b'K' b'b'",
    "Kr Ka": "Krypton - b'K' b'a'",
    "K Ka": "Potash - b'K' b'a'",
    "Ir Lb": "Iridium - b'L' b'b'",
    "Ir La": "Iridium - b'L' b'a'",
    "In Kb": "Indium - b'K' b'b'",
    "In Ka": "Indium - b'K' b'a'",
    "Hg Lb": "Mercury - b'L' b'b'",
    "Hg La": "Mercury - b'L' b'a'",
    "Ga Kb": "Gallium - b'K' b'b'",
    "Ga Ka": "Gallium - b'K' b'a'",
    "Fe Kb": "Iron - b'K' b'b'",
    "Fe Ka": "Iron - b'K' b'a'",
    "Cu Kb": "Copper - b'K' b'b'",
    "Cu Ka": "Copper - b'K' b'a'",
    "Cr Kb": "Chromium - b'K' b'b'",
    "Cr Ka": "Chromium - b'K' b'a'",
    "Co Kb": "Cobalt - b'K' b'b'",
    "Co Ka": "Cobalt - b'K' b'a'",
    "Cd Ka": "Cadmium - b'K' b'a'",
    "Ca Kb": "Calcium - b'K' b'b'",
    "Ca Ka": "Calcium - b'K' b'a'",
    "Br Kb": "Bromine - b'K' b'b'",
    "Br Ka": "Bromine - b'K' b'a'",
    "As Kb": "Arsenic - b'K' b'b'",
    "As Ka": "Arsenic - b'K' b'a'",
    "Al Ka": "Aluminum - b'K' b'a'",
    "Ag Ka": "Silver - b'K' b'a'",
}

def _safe_find_text(parent, tag, default=''):
    """Helper function to safely find text from an XML element."""
    if parent is None:
        return default
    f = parent.find(tag) if isinstance(tag, str) else parent
    if f is not None and f.text:
        return f.text.strip()
    return default

def _parse_spectrum_results(sp):
    """
    Returns a dict of short_label -> float(value) from <spectrumResult> blocks.
    """
    d = {}
    sr = sp.find('spectrumResults')
    if sr is None:
        return d
    for x in sr.findall('spectrumResult'):
        target_raw = _safe_find_text(x, 'target', '')
        val_str = _safe_find_text(x, 'value', '0')
        try:
            val = float(val_str)
        except:
            val = 0.0

        short_label = target_raw.split('/')[0].strip()  # e.g. "Pt La / cps" -> "Pt La"
        d[short_label] = val

    return d

def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    company = _safe_find_text(root, './/Company')
    location = _safe_find_text(root, './/Location')
    wellid = _safe_find_text(root, './/WellID')

    # The main XRF results block
    xrfresults = root.find('.//XRFresults')
    if xrfresults is None:
        return pd.DataFrame(), wellid

    xrfresult_list = xrfresults.findall('XrfResult')
    if not xrfresult_list:
        return pd.DataFrame(), wellid

    rows = []

    for xr in xrfresult_list:
        line_id = _safe_find_text(xr, 'lineId')
        meas_id = _safe_find_text(xr, 'measurementId')

        start_pos = float(_safe_find_text(xr.find('./startTime'), 'position', '0'))
        stop_pos  = float(_safe_find_text(xr.find('./stopTime'), 'position', '0'))
        dist = stop_pos - start_pos   # total distance for this block

        spectra_el = xr.find('spectra')
        if spectra_el is None:
            continue

        # Collect all <spectrum>, sort by <timeStart> to ensure chronological
        spectra_list = []
        for s in spectra_el.findall('spectrum'):
            t_start_str = _safe_find_text(s, 'timeStart', '')
            spectra_list.append((s, t_start_str))

        spectra_list.sort(key=lambda x: x[1])  # lexical sort
        n_spectra = len(spectra_list)
        if n_spectra == 0:
            continue

        # Calculate equal depth increment for each spectrum
        depth_increment = dist / n_spectra if n_spectra > 0 else 0

        for i, (spectrum_el, _) in enumerate(spectra_list):
            # Equal distribution of depths
            d_from = start_pos + (i * depth_increment)
            d_to = start_pos + ((i + 1) * depth_increment)
            
            # Ensure last spectrum exactly reaches stop_pos to avoid rounding errors
            if i == n_spectra - 1:
                d_to = stop_pos

            interval_length = d_to - d_from

            row_data = {
                'Company': company,
                'Location': location,
                'WellID': wellid,
                'Measurement ID': meas_id,
                'Line ID': line_id,
                'Depth From': d_from,
                'Depth To':   d_to
            }

            # Mark '!' if zero-length interval
            if interval_length == 0:
                row_data['WellID'] += '!'
            else:
                # Mark '*' if acquisition time is less than 10 seconds
                # Look for timeAcquisition element in the spectrum
                time_acq_str = _safe_find_text(spectrum_el, 'timeAcquisition', '0')
                try:
                    time_acquisition = float(time_acq_str)
                    if time_acquisition < 10:
                        row_data['WellID'] += '*'
                except ValueError:
                    # If we can't parse the time acquisition, don't add the mark
                    pass

            # parse intensities for this spectrum
            spec_map = _parse_spectrum_results(spectrum_el)
            for short_label, val in spec_map.items():
                if short_label in SHORT_LABEL_MAP:
                    row_data[SHORT_LABEL_MAP[short_label]] = val

            rows.append(row_data)

    if not rows:
        return pd.DataFrame(), wellid

    df = pd.DataFrame(rows)
    base_cols = ['Company','Location','WellID','Measurement ID','Line ID','Depth From','Depth To']
    extra_cols = [c for c in df.columns if c not in base_cols]
    extra_cols.sort()
    df = df.reindex(columns=base_cols + extra_cols)
    return df, wellid

def clean_and_combine_xml_data(xml_dir, output_file, required_columns):
    frames = []
    wellids = set()
    
    for f in os.listdir(xml_dir):
        if f.lower().endswith('.xml'):
            path = os.path.join(xml_dir, f)
            df, wellid = parse_xml(path)
            if not df.empty:
                # ensure required_columns exist
                for c in required_columns:
                    if c not in df.columns:
                        df[c] = None
                df = df[required_columns]
                frames.append(df)
                if wellid:
                    wellids.add(wellid)
    
    if not frames:
        combined = pd.DataFrame(columns=required_columns)
    else:
        combined = pd.concat(frames, ignore_index=True)
    
    # Only save to file if output_file is provided
    if output_file:
        # Check if output file exists and append if it does
        if os.path.exists(output_file):
            existing_df = pd.read_excel(output_file)
            combined = pd.concat([existing_df, combined], ignore_index=True)
            # Remove potential duplicates
            combined = combined.drop_duplicates(subset=['WellID', 'Measurement ID', 'Line ID', 'Depth From', 'Depth To'])
        
        combined.to_excel(output_file, index=False)
    
    # Return the combined DataFrame and a representative wellid
    representative_wellid = next(iter(wellids)) if wellids else "Unknown"
    return combined, representative_wellid

def reorder_columns(output_file, required_columns):
    df = pd.read_excel(output_file)
    # Only select columns that exist in the DataFrame
    existing_columns = [col for col in required_columns if col in df.columns]
    df = df[existing_columns]
    if 'Depth From' in df.columns:
        df = df.sort_values('Depth From')
    df.to_excel(output_file, index=False)
    return df

def apply_data_bars(file_path):
    df = pd.read_excel(file_path)
    wb = openpyxl.load_workbook(file_path)
    ws = wb.active
    start_col = 7
    end_col = df.shape[1]
    col_list = df.columns[start_col:end_col]
    color_map = {}

    for c in col_list:
        base = c.split(' - ')[0]
        if base not in color_map:
            color_map[base] = _rand_hex()

    for i, c in enumerate(col_list, start=start_col + 1):
        vals = df[c].dropna()
        if vals.empty:
            continue
        mn, mx = vals.min(), vals.max()
        if mn == mx:
            continue
        base = c.split(' - ')[0]
        color = color_map[base]
        rule = DataBarRule(
            start_type='num', start_value=mn,
            end_type='num', end_value=mx,
            color=Color(rgb=color),
            showValue="None"
        )
        letter = openpyxl.utils.get_column_letter(i)
        ws.conditional_formatting.add(f"{letter}2:{letter}{ws.max_row}", rule)

    wb.save(file_path)

def _rand_hex():
    return ''.join(f"{random.randint(0,255):02X}" for _ in range(3))

name_normalization = {
    "sulphur": "sulphur", "sulfur": "sulphur",
    "molybdenum": "molybdenum", "molybdenium": "molybdenum",
    "aluminium": "aluminum", "aluminum": "aluminum",
    "caesium": "cesium", "cesium": "cesium", "titanium": "titanum",
}

def parse_calib_file(calib_fp):
    with open(calib_fp, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    return [(lines[i], lines[i+1]) for i in range(0, len(lines), 2)]

def process_formula(formula):
    if '=' in formula:
        formula = formula.split('=',1)[1].strip()
    formula = re.sub(r'(\d)\(', r'\1*(', formula)
    formula = formula.replace('×','*')
    formula = re.sub(r'10\^−?(\d+)', r'10**(-\1)', formula)
    formula = re.sub(r'10\^(\d+)', r'10**(\1)', formula)
    return formula

def safe_var(name):
    return re.sub(r'\W+', '_', name)

def map_var_to_col(var_text):
    m = re.match(r'([\w\s]+)\s*([KL][ABab])', var_text, re.IGNORECASE)
    if not m:
        return var_text
    elem, suffix = m.group(1).strip(), m.group(2).upper()
    norm = name_normalization.get(elem.lower(), elem.lower())
    return f"{norm.title()} - b'{suffix[0]}' b'{suffix[1].lower()}'"

def build_formula_expr(formula):
    var_map = {}
    pattern = re.compile(r'([A-Za-z]+(?:\s+[A-Za-z]+)*\s*[KL][ABab])')
    def repl(m):
        vt = m.group(1).strip()
        col = map_var_to_col(vt)
        safe = safe_var(col)
        var_map[vt] = {"safe": safe, "excel": col}
        return safe
    proc = pattern.sub(repl, formula)
    return proc, var_map

def fallback_brit_am(row, col):
    m = re.match(r'([A-Za-z]+)\s*-\s*b\'[KL]\'\s*b\'[ab]\'', col)
    if not m:
        return None
    elem = m.group(1)
    for key, canon in name_normalization.items():
        if canon.title() == elem:
            alt = col.replace(elem, key.title(), 1)
            val = row.get(alt, None)
            if val is not None:
                return val
    return None

def fallback_k_l(row, col):
    m = re.match(r'(.*b\')[KL](\' b\'[ab]\')', col)
    if not m:
        return None
    alt = col.replace("K", "L", 1) if "K" in col else col.replace("L", "K", 1)
    return row.get(alt, None)

def eval_calibration(df, new_col, expr, var_map):
    def row_eval(row):
        loc = {}
        for orig, info in var_map.items():
            col = info["excel"]
            val = row.get(col, None)
            if val is None:
                val = fallback_brit_am(row, col)
            if val is None:
                val = fallback_k_l(row, col)
            try:
                val = float(val)
            except (ValueError, TypeError):
                val = 0
            loc[info["safe"]] = val
        try:
            return eval(expr, {"__builtins__": None}, loc)
        except Exception:
            return None
    df[new_col] = df.apply(row_eval, axis=1)

def process_calibrated_file(xrf_file, calib_file, output_file):
    """Process XRF file with calibration formulas, preserve original with additions"""
    if not os.path.exists(xrf_file) or not os.path.exists(calib_file):
        print(f"Error: Missing input files - XRF: {xrf_file}, Calibration: {calib_file}")
        return None
        
    # Check if output file already exists and load it if so
    if os.path.exists(output_file):
        df = pd.read_excel(output_file)
        print(f"Updating existing calibrated file: {output_file}")
    else:
        df = pd.read_excel(xrf_file)
        print(f"Creating new calibrated file: {output_file}")
    
    if "Depth To" not in df.columns:
        print("Error: 'Depth To' column not found in XRF file")
        return None
    
    # Remove '*' marker from WellID for calibrated file
    if 'WellID' in df.columns:
        df['WellID'] = df['WellID'].astype(str).str.replace('*', '', regex=False)
    
    # Load and apply calibrations
    calib_entries = parse_calib_file(calib_file)
    print(f"Found {len(calib_entries)} calibration formulas")
    
    # Keep track of calibration column names
    calibration_columns = []
    
    # Apply each calibration formula to the DataFrame
    for new_col, formula_line in calib_entries:
        print(f"Processing calibration: {new_col}")
        expr_clean = process_formula(formula_line)
        expr, var_map = build_formula_expr(expr_clean)
        eval_calibration(df, new_col, expr, var_map)
        calibration_columns.append(new_col)
    
    # Replace negative values with 0 in calibration columns
    for col in calibration_columns:
        if col in df.columns:
            # Replace negative values with 0, preserve NaN/None values
            df[col] = df[col].apply(lambda x: 0 if pd.notnull(x) and x < 0 else x)
            negative_count = (df[col] < 0).sum()
            if negative_count > 0:
                print(f"Replaced {negative_count} negative values with 0 in column: {col}")
    
    # Reorder columns: base columns + calibration columns + remaining columns
    base_cols = ['Company', 'Location', 'WellID', 'Measurement ID', 'Line ID', 'Depth From', 'Depth To']
    
    # Get existing calibration columns that are actually in the dataframe
    existing_calib_cols = [col for col in calibration_columns if col in df.columns]
    
    # Get all remaining columns (original XRF intensity columns)
    remaining_cols = [col for col in df.columns if col not in base_cols and col not in existing_calib_cols]
    
    # Reorder the dataframe
    final_column_order = base_cols + existing_calib_cols + remaining_cols
    df = df[final_column_order]
    
    # Save the updated DataFrame
    df.to_excel(output_file, index=False)
    print(f"Saved calibrated data to: {output_file}")
    
    # Apply data bars to the calibrated file
    apply_data_bars(output_file)
    
    return output_file

def run_main(base_folder, xml_path, calib_path):
    # Determine XML source directory and output directory
    if xml_path and os.path.exists(xml_path):
        if os.path.isdir(xml_path):
            xml_dir = xml_path
        else:
            xml_dir = os.path.dirname(xml_path)
        # Create output directory within the XML directory
        output_dir = os.path.join(xml_dir, "Output")
    else:
        # Fallback to XML folder within base_folder
        xml_dir = os.path.join(base_folder, "XML")
        if not os.path.exists(xml_dir):
            messagebox.showerror("Error", "No XML files found. Please provide a valid XML path.")
            return
        # Create output directory within the base folder
        output_dir = os.path.join(base_folder, "Output")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Using XML files from: {xml_dir}")
    print(f"Output files will be saved to: {output_dir}")
    
    # Define required columns - if new elements added have to be updated in the list
    req_cols = [
        'Company', 'Location', 'WellID', 'Measurement ID', 'Line ID', 'Depth From', 'Depth To',
        "Magnesium - b'K' b'a'", "Aluminum - b'K' b'a'", "Silicon - b'K' b'a'", "Phosphorus - b'K' b'a'",
        "Sulfur - b'K' b'a'", "Chlorine - b'K' b'a'", "Argon - b'K' b'a'", "Potash - b'K' b'a'", "Potash - b'K' b'b'",
        "Calcium - b'K' b'a'", "Calcium - b'K' b'b'", "Scandium - b'K' b'a'", "Scandium - b'K' b'b'",
        "Titanum - b'K' b'a'", "Titanum - b'K' b'b'", "Vanadium - b'K' b'a'", "Vanadium - b'K' b'b'",
        "Chromium - b'K' b'a'", "Chromium - b'K' b'b'", "Manganese - b'K' b'a'", "Manganese - b'K' b'b'",
        "Iron - b'K' b'a'", "Iron - b'K' b'b'", "Cobalt - b'K' b'a'", "Cobalt - b'K' b'b'", "Nickel - b'K' b'a'",
        "Nickel - b'K' b'b'", "Copper - b'K' b'a'", "Copper - b'K' b'b'", "Zinc - b'K' b'a'", "Zinc - b'K' b'b'",
        "Gallium - b'K' b'a'", "Gallium - b'K' b'b'", "Germanium - b'K' b'a'", "Germanium - b'K' b'b'",
        "Arsenic - b'K' b'a'", "Arsenic - b'K' b'b'", "Selenium - b'K' b'a'", "Selenium - b'K' b'b'",
        "Bromine - b'K' b'a'", "Bromine - b'K' b'b'", "Krypton - b'K' b'a'", "Krypton - b'K' b'b'",
        "Strontium - b'K' b'a'", "Strontium - b'K' b'b'", "Yttrium - b'K' b'a'", "Yttrium - b'K' b'b'",
        "Zirconium - b'K' b'a'", "Zirconium - b'K' b'b'", "Niobium - b'K' b'a'", "Niobium - b'K' b'b'",
        "Molybdenium - b'K' b'a'", "Molybdenium - b'K' b'b'", "Rhodium - b'K' b'a'", "Rhodium - b'K' b'b'",
        "Palladium - b'K' b'a'", "Palladium - b'K' b'b'", "Silver - b'K' b'a'", "Silver - b'K' b'b'",
        "Cadmium - b'K' b'a'", "Cadmium - b'K' b'b'", "Indium - b'K' b'a'", "Indium - b'K' b'b'",
        "Tin - b'K' b'a'", "Tin - b'K' b'b'", "Antimony - b'K' b'a'", "Antimony - b'K' b'b'",
        "Tellurium - b'K' b'a'", "Tellurium - b'K' b'b'", "Iodine - b'L' b'a'", "Iodine - b'L' b'b'",
        "Xenon - b'L' b'a'", "Xenon - b'L' b'b'", "Tantalum - b'L' b'a'", "Tantalum - b'L' b'b'",
        "Tungsten - b'L' b'a'", "Tungsten - b'L' b'b'", "Rhenium - b'L' b'a'", "Rhenium - b'L' b'b'",
        "Osmium - b'L' b'a'", "Osmium - b'L' b'b'", "Iridium - b'L' b'a'", "Iridium - b'L' b'b'",
        "Platinum - b'L' b'a'", "Platinum - b'L' b'b'", "Platin - b'L' b'a'", "Platin - b'L' b'b'",
        "Mercury - b'L' b'a'", "Mercury - b'L' b'b'", "Thallium - b'L' b'a'", "Thallium - b'L' b'b'",
        "Lead - b'L' b'a'", "Lead - b'L' b'b'", "Silicon - b'K' b'b'", "Sulfur - b'K' b'b'",
        "Sodium - b'K' b'a'"
    ]
    
    # Process XML files directly to the wellid-based filename
    print("Processing XML files and generating XRF data...")
    
    # Get combined data and wellid, without creating any intermediate file  
    frames = []
    wellids = set()
    
    # Go through all XML files
    for f in os.listdir(xml_dir):
        if f.lower().endswith('.xml'):
            path = os.path.join(xml_dir, f)
            df, wellid = parse_xml(path)
            if not df.empty:
                # Ensure required columns
                for c in req_cols:
                    if c not in df.columns:
                        df[c] = None
                df = df[req_cols]
                frames.append(df)
                if wellid:
                    wellids.add(wellid)

    # Combine all parsed frames
    if not frames:
        combined_df = pd.DataFrame(columns=req_cols)
        wellid = "OutputSample"  # fallback if NO data found
    else:
        combined_df = pd.concat(frames, ignore_index=True)
        if 'Depth From' in combined_df.columns:
            combined_df = combined_df.sort_values('Depth From')
        wellid = next(iter(wellids)) if wellids else "OutputSample"  # fallback if no wellid

    # Clean and validate wellid
    if not wellid or wellid.strip().lower() in ["", "unknown", "unknownwell", "none"]:
        wellid = "OutputSample"
    else:
        wellid = wellid.replace(" ", "_").replace(".", "_")
    
    # Create the base XRF Excel file with the wellid in the name
    base_xrf_file = os.path.join(output_dir, f"XRF_data_{wellid}.xlsx")
    
    # Check if the file already exists
    if os.path.exists(base_xrf_file):
        existing_df = pd.read_excel(base_xrf_file)
        combined_df = pd.concat([existing_df, combined_df], ignore_index=True)
        # Remove potential duplicates
        combined_df = combined_df.drop_duplicates(subset=['WellID', 'Measurement ID', 'Line ID', 'Depth From', 'Depth To'])
    
    # Save directly to the wellid file
    combined_df.to_excel(base_xrf_file, index=False)
    
    # Apply data bars to the base file
    apply_data_bars(base_xrf_file)
    print(f"Base XRF data file created: {base_xrf_file}")
    
    # Apply calibration if a calibration file is provided
    if calib_path and os.path.exists(calib_path):
        print(f"Applying calibration from: {calib_path}")
        calibrated_file = os.path.join(output_dir, f"XRF_Data_Calibrated_{wellid}.xlsx")
        result = process_calibrated_file(base_xrf_file, calib_path, calibrated_file)
        if result:
            print(f"Calibrated XRF file created: {calibrated_file}")
        else:
            print("Calibration process failed.")
    else:
        print("No calibration file provided or file not found. Skipping calibration.")
    
    messagebox.showinfo("Success", f"XRF data processing completed. Files saved to {output_dir}")
    return output_dir

def start_processing_xml_combine():
    # Get input paths from GUI entries
    base_folder = folder_entry.get().strip()
    xml_path = xml_entry.get().strip()
    calib_path = xrfcalib_entry.get().strip()
    
    # Validate inputs
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return
    
    # Validate calibration file if provided
    if calib_path and not os.path.isfile(calib_path):
        messagebox.showerror("Error", "Please provide a valid calibration file path.")
        return
    
    # Redirect stdout to log text widget if available
    old_stdout = sys.stdout
    if 'log_text' in globals():
        sys.stdout = TextRedirector(log_text, "stdout")
    
    try:
        # Run the main processing function
        run_main(base_folder, xml_path, calib_path)
    except Exception as e:
        error_msg = f"An error occurred during XRF processing:\n{str(e)}"
        print(error_msg)
        messagebox.showerror("Error", error_msg)
    finally:
        # Restore stdout
        sys.stdout = old_stdout





# CODE 6: Extract RGB Core Pieces with Enhancement

def enhance_core_box_image_fast(image):
    """
    Faster enhancement with fewer operations while maintaining quality.
    Same function as in Code 8 for consistency.
    """
    # Method 1: Combined brightness/contrast in one operation
    # Slightly increase contrast and brightness
    enhanced = cv2.addWeighted(image, 1.15, image, 0, 25)
    
    # Method 2: Simple gamma correction using lookup table
    gamma = 0.8
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    enhanced = cv2.LUT(enhanced, table)
    
    return enhanced

def estimate_jpeg_quality_for_size(original_size):
    """
    Estimate JPEG quality based on original file size.
    Same function as in Code 8 for consistency.
    """
    size_mb = original_size / (1024 * 1024)
    
    if size_mb < 2:
        return 80
    elif size_mb < 5:
        return 85
    else:
        return 90

def extract_rgb_core_pieces():
    base_folder = folder_entry.get().strip()
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return

    # Check for XML folder; if empty, fallback to xml_entry
    xml_dir = os.path.join(base_folder, "XML")
    if not os.path.exists(xml_dir) or not os.listdir(xml_dir):
        xml_entry_path = xml_entry.get().strip()
        if not xml_entry_path or not os.path.exists(xml_entry_path):
            messagebox.showerror("Error", "XML files are missing.")
            return
        else:
            xml_dir = xml_entry_path

    # Check for RGB Core Box Images folder
    rgb_core_box_dir = os.path.join(base_folder, "RGB Core Box Images")
    if not os.path.exists(rgb_core_box_dir) or not os.listdir(rgb_core_box_dir):
        messagebox.showerror("Error", "RGB Core Box Images folder is missing or empty.")
        return

    # First try Masked_Coreg_HSI
    swir_dir = os.path.join(base_folder, "Masked_Coreg_HSI")
    if not os.path.exists(swir_dir) or not os.listdir(swir_dir):
        # If SWIR_HSI_Files is missing/empty, try Corrected_SWIR
        corrected_swir = os.path.join(base_folder, "Corrected_SWIR")
        if not os.path.exists(corrected_swir) or not os.listdir(corrected_swir):
            messagebox.showerror("Error", "Masked_Coreg_HSI and Corrected_SWIR are both missing or empty.")
            return
        else:
            swir_dir = corrected_swir

    # Create output folder if not existing
    output_dir = os.path.join(base_folder, "RGB Core Pieces")
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    txt_file_path = os.path.join(base_folder, "core_images_list.txt")

    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    try:
        with open(txt_file_path, "a") as txt_file:
            for rgb_filename in os.listdir(rgb_core_box_dir):
                if not rgb_filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                if "_ext" in rgb_filename:
                    print(f"Skipping already processed file: {rgb_filename}")
                    sys.stdout.flush()
                    continue

                rgb_image_path = os.path.join(rgb_core_box_dir, rgb_filename)
                print(f"\nProcessing RGB image: {rgb_filename}")
                sys.stdout.flush()

                # Attempt to find a UUID in { ... } brackets
                uuid_match = re.search(r"\{.*?\}", rgb_filename)
                if not uuid_match:
                    print(f"  [WARN] No valid UUID found in {rgb_filename}. Skipping.")
                    sys.stdout.flush()
                    continue
                uuid = uuid_match.group(0)

                # Try matching an XML
                matching_xml_files = [
                    f for f in os.listdir(xml_dir)
                    if uuid in f and f.lower().endswith('.xml')
                ]
                if not matching_xml_files:
                    print(f"  [WARN] No XML file found for UUID {uuid}. Skipping.")
                    sys.stdout.flush()
                    continue

                xml_file_path = os.path.join(xml_dir, matching_xml_files[0])
                print(f"  Found XML: {os.path.basename(xml_file_path)}")
                sys.stdout.flush()
                try:
                    tree = ET.parse(xml_file_path)
                except Exception as e:
                    print(f"  [ERROR] Could not parse {xml_file_path}: {e}")
                    sys.stdout.flush()
                    continue

                root_xml = tree.getroot()
                image2d_el = root_xml.find('.//image2D')
                if image2d_el is not None:
                    dims_el = image2d_el.find('dimensions')
                    if dims_el is not None:
                        image2d_width_m = float(dims_el.get('x','1.0'))
                        image2d_height_m= float(dims_el.get('y','1.0'))
                    else:
                        image2d_width_m, image2d_height_m=1.0,1.0

                    transform_el = image2d_el.find('transform')
                    if transform_el is not None:
                        translate_el= transform_el.find('translate')
                        if translate_el is not None:
                            image2d_trans_x= float(translate_el.get('x','0.0'))
                            image2d_trans_y= float(translate_el.get('y','0.0'))
                        else:
                            image2d_trans_x,image2d_trans_y=0.0,0.0
                    else:
                        image2d_trans_x,image2d_trans_y=0.0,0.0
                else:
                    image2d_width_m,image2d_height_m=1.0,1.0
                    image2d_trans_x,image2d_trans_y=0.0,0.0

                core_box_def = root_xml.find('.//CoreBoxDef')
                if core_box_def is None:
                    print(f"  [ERROR] No <CoreBoxDef> in {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                corebox_trans_x= float(core_box_def.get('Translation-X','0.0'))
                corebox_trans_y= float(core_box_def.get('Translation-Y','0.0'))
                box_rot_deg=    float(core_box_def.get('Rotation','0'))
                box_scaling_x=  float(core_box_def.get('ScalingX','1'))
                box_scaling_y=  float(core_box_def.get('ScalingY','1'))

                box_def_el= core_box_def.find('BoxDefinition')
                if box_def_el is None:
                    print(f"  [ERROR] No <BoxDefinition> in {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                ext_dims_el= box_def_el.find('ExternalDims')
                if ext_dims_el is None:
                    print(f"  [ERROR] No <ExternalDims> in {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                ext_width_m=  float(ext_dims_el.get('Width','1.0'))
                ext_height_m= float(ext_dims_el.get('Height','1.0'))
                external_width=  float(box_def_el.get('ExternalWidth','0.0'))
                external_height= float(box_def_el.get('ExternalHeight','0.0'))
                internal_width=  float(box_def_el.get('InternalWidth','0.0'))
                num_compartments= int(box_def_el.get('NumCompartments','1'))

                final_box_width_m  = ext_width_m  * box_scaling_x
                final_box_height_m = ext_height_m * box_scaling_y

                core_pieces_el= core_box_def.find('.//CorePieces')
                if core_pieces_el is None:
                    print(f"  [ERROR] No <CorePieces> in {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                piece_list= core_pieces_el.findall('CorePiece')
                if not piece_list:
                    print(f"  [ERROR] No <CorePiece> entries in {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                final_trans_x= corebox_trans_x - image2d_trans_x
                final_trans_y= corebox_trans_y - image2d_trans_y

                # Load original image
                image_cv_original = cv2.imread(rgb_image_path)
                if image_cv_original is None:
                    print(f"  [ERROR] Could not load image: {rgb_image_path}")
                    sys.stdout.flush()
                    continue

                # Get original file size for quality estimation
                original_size = os.path.getsize(rgb_image_path)
                
                # Apply enhancement to a copy of the image
                print(f"  Applying enhancement...")
                sys.stdout.flush()
                image_cv_enhanced = enhance_core_box_image_fast(image_cv_original.copy())
                
                # Create a temporary enhanced version with reduced quality
                estimated_quality = estimate_jpeg_quality_for_size(original_size)
                _, temp_buffer = cv2.imencode('.jpg', image_cv_enhanced, [cv2.IMWRITE_JPEG_QUALITY, estimated_quality])
                image_cv = cv2.imdecode(temp_buffer, cv2.IMREAD_COLOR)
                
                print(f"  Enhanced with quality setting: {estimated_quality}")
                sys.stdout.flush()

                img_h, img_w= image_cv.shape[:2]
                with Image.open(rgb_image_path) as pil_img:
                    xdpi, ydpi= pil_img.info.get('dpi',(96,96))

                scale_x= img_w/ image2d_width_m
                scale_y= img_h/ image2d_height_m
                print(f"  Image dimensions: {img_w}x{img_h} px; Scale: ({scale_x:.3f}, {scale_y:.3f}) px/m")
                sys.stdout.flush()

                def rotate_point(x_m, y_m, pivot_x_m, pivot_y_m, deg):
                    if abs(deg)<1e-9:
                        return x_m, y_m
                    rad= math.radians(deg)
                    dx= x_m- pivot_x_m
                    dy= y_m- pivot_y_m
                    rx= dx*math.cos(rad)- dy*math.sin(rad)
                    ry= dx*math.sin(rad)+ dy*math.cos(rad)
                    return pivot_x_m+rx, pivot_y_m+ry

                def box_local_to_pixels(x_m, y_m):
                    world_x= final_trans_x+ x_m
                    world_y= final_trans_y+ y_m
                    rx_m, ry_m= rotate_point(world_x, world_y, final_trans_x, final_trans_y, box_rot_deg)
                    px= rx_m* scale_x
                    py= ry_m* scale_y
                    return px, py

                def row_top_bottom(i):
                    usable_vertical= final_box_height_m - 2*external_height - (num_compartments-1)*internal_width
                    top_m= external_height+ i*(usable_vertical/num_compartments+ internal_width)
                    bot_m= top_m+ (usable_vertical/num_compartments)
                    return top_m,bot_m

                usable_horizontal= final_box_width_m - 2* external_width
                if usable_horizontal<0:
                    print(f"  [ERROR] Not enough horizontal space in box definition for {xml_file_path}. Skipping.")
                    sys.stdout.flush()
                    continue

                cropped_pieces= []
                for idx,piece_el in enumerate(piece_list):
                    try:
                        length_rel= float(piece_el.get('LengthRel','1.0'))
                        offset_rel= float(piece_el.get('OffsetRel','0.0'))
                        row_idx= int(piece_el.get('Row','0'))
                    except Exception as e:
                        print(f"  [WARN] Error reading CorePiece {idx}: {e}")
                        sys.stdout.flush()
                        continue

                    top_m, bot_m= row_top_bottom(row_idx)
                    piece_left_m= external_width+ offset_rel* usable_horizontal
                    piece_width_m= length_rel* usable_horizontal
                    piece_right_m= piece_left_m+ piece_width_m

                    tl_px,tl_py= box_local_to_pixels(piece_left_m, top_m)
                    br_px,br_py= box_local_to_pixels(piece_right_m, bot_m)

                    x0,y0= int(round(tl_px)), int(round(tl_py))
                    x1,y1= int(round(br_px)), int(round(br_py))

                    x_min, x_max= min(x0,x1), max(x0,x1)
                    y_min, y_max= min(y0,y1), max(y0,y1)
                    w_px= x_max- x_min
                    h_px= y_max- y_min

                    if x_min<0 or y_min<0 or x_max>img_w or y_max>img_h:
                        print(f"  [WARN] Piece {idx} out of image bounds. Skipping.")
                        sys.stdout.flush()
                        continue
                    if w_px<2 or h_px<2:
                        print(f"  [WARN] Piece {idx} too small: {w_px}x{h_px} px. Skipping.")
                        sys.stdout.flush()
                        continue

                    # Extract from the enhanced image
                    crop_img= image_cv[y_min:y_max, x_min:x_max]
                    cropped_pieces.append((idx,crop_img))
                    print(f"  Extracted piece {idx}: bounds=({x_min},{y_min}) to ({x_max},{y_max}).")
                    sys.stdout.flush()

                if not cropped_pieces:
                    print(f"  [WARN] No valid pieces extracted from {rgb_filename}.")
                    sys.stdout.flush()
                    continue

                # find matching SWIR folder => rename pieces
                hsi_folder=None
                for folder in os.listdir(swir_dir):
                    if uuid in folder:
                        hsi_folder= os.path.join(swir_dir, folder)
                        break
                if hsi_folder is None:
                    print(f"  [WARN] No matching HSI folder for UUID {uuid}. Skipping renaming.")
                    sys.stdout.flush()
                    continue

                hsi_files= [f for f in os.listdir(hsi_folder) if f.lower().endswith('.hdr')]
                if not hsi_files:
                    print(f"  [WARN] No .hdr files found in HSI folder {hsi_folder}.")
                    sys.stdout.flush()
                    continue

                try:
                    hsi_files_sorted= sorted(hsi_files, key=lambda x: float(re.match(r"([\d\.]+)", x).group(1)))
                except Exception as e:
                    print(f"  [ERROR] Sorting HSI files failed in {hsi_folder}: {e}")
                    sys.stdout.flush()
                    continue

                if len(hsi_files_sorted)!= len(cropped_pieces):
                    print(f"  [WARN] #HSI files={len(hsi_files_sorted)} != #cropped pieces={len(cropped_pieces)}. Skipping.")
                    sys.stdout.flush()
                    continue

                for (idx,piece_cv),hsi_filename in zip(cropped_pieces,hsi_files_sorted):
                    piece_pil= Image.fromarray(cv2.cvtColor(piece_cv,cv2.COLOR_BGR2RGB))
                    rotated= piece_pil.rotate(-90, expand=True)
                    flipped = rotated.transpose(Image.FLIP_LEFT_RIGHT)
                    
                    new_w,new_h= flipped.size
                    resized= flipped.resize((int(new_w*0.25),int(new_h*0.25)))
                    
                    base_hsi= os.path.splitext(hsi_filename)[0]
                    parts= base_hsi.split('_')
                    
                    
                    if len(parts) >= 2:
                        try:
                            depth_from = float(parts[0])
                            depth_to = float(parts[1])
                        except:
                            print(f"  [ERROR] Depth parse failed for {hsi_filename}")
                            sys.stdout.flush()
                            continue
                    else:
                        print(f"  [WARN] Unexpected HSI filename format: {hsi_filename}. Skipping piece {idx}.")
                        sys.stdout.flush()
                        continue
                    
                    # Use the UUID from the original RGB filename, not from HSI filename
                    formatted_depth_from= f"{depth_from:.3f}"
                    formatted_depth_to  = f"{depth_to:.3f}"
                    new_name= f"{uuid}_{formatted_depth_from}_{formatted_depth_to}.jpg"
                    new_path= os.path.join(output_dir,new_name)
                    resized.save(new_path)
                    txt_file.write(f"{formatted_depth_from}\t{formatted_depth_to}\t{new_path}\n")
                    print(f"  Saved processed piece {idx} => {new_name}")
                    sys.stdout.flush()

                # Rename the ORIGINAL file 
                base, ext= os.path.splitext(rgb_filename)
                new_rgb_filename= base+"_ext"+ ext
                new_rgb_image_path= os.path.join(rgb_core_box_dir,new_rgb_filename)
                os.rename(rgb_image_path,new_rgb_image_path)
                print(f"  Marked RGB file as processed: {new_rgb_filename} (original image preserved)")
                sys.stdout.flush()

        print("\nAll files processed. Core pieces list updated in", txt_file_path)
        messagebox.showinfo("Success", "RGB Core Piece Images extracted and saved successfully.")
        sys.stdout.flush()
        

    except Exception as e:
        messagebox.showerror("Error", f"An error occurred during RGB extraction:\n{e}")
    finally:
        sys.stdout = old_stdout




# CODE 7: Make Depth Overlap Corrections

def parse_corebox_info_overlap(xml_path):
    """Extract CoreBox number and depth information from XML"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Get CoreBox number
        corebox_elem = root.find('.//CoreBox')
        if corebox_elem is None:
            print(f"Warning: No CoreBox found in {xml_path}")
            return None
        
        corebox_num = int(corebox_elem.text)
        
        # Get measurement ID
        measurement_id_elem = root.find('.//measurement/id')
        measurement_id = None
        if measurement_id_elem is not None:
            measurement_id = measurement_id_elem.text.strip('{}')
        
        # Get HSI results
        hsi_results = root.find('.//HSIresults')
        if hsi_results is None:
            print(f"Warning: No HSIresults in {xml_path}")
            return None
        
        hsi_pieces = []
        for hsi in hsi_results.findall('HSIresult'):
            start_elem = hsi.find('startTime/position')
            stop_elem = hsi.find('stopTime/position')
            if start_elem is not None and stop_elem is not None:
                start_pos = float(start_elem.text)
                stop_pos = float(stop_elem.text)
                hsi_pieces.append((start_pos, stop_pos))
        
        if not hsi_pieces:
            print(f"Warning: No HSI pieces found in {xml_path}")
            return None
        
        # Sort pieces by start position
        hsi_pieces.sort(key=lambda x: x[0])
        
        return {
            'path': xml_path,
            'corebox': corebox_num,
            'measurement_id': measurement_id,
            'first_start': hsi_pieces[0][0],
            'last_stop': hsi_pieces[-1][1],
            'pieces': hsi_pieces
        }
        
    except Exception as e:
        print(f"Error parsing {xml_path}: {e}")
        return None

def sort_hsi_results_in_xml_overlap(xml_path):
    """Sort HSI and VNIR results by start position within the XML"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        modified = False
        
        # Sort HSIresults
        hsi_results = root.find('.//HSIresults')
        if hsi_results is not None:
            hsi_list = list(hsi_results.findall('HSIresult'))
            if len(hsi_list) > 1:
                hsi_with_pos = []
                for hsi in hsi_list:
                    start_elem = hsi.find('startTime/position')
                    if start_elem is not None:
                        pos = float(start_elem.text)
                        hsi_with_pos.append((pos, hsi))
                
                hsi_with_pos.sort(key=lambda x: x[0])
                
                original_order = [float(hsi.find('startTime/position').text) for hsi in hsi_list if hsi.find('startTime/position') is not None]
                sorted_order = [x[0] for x in hsi_with_pos]
                
                if original_order != sorted_order:
                    for hsi in hsi_list:
                        hsi_results.remove(hsi)
                    for _, hsi in hsi_with_pos:
                        hsi_results.append(hsi)
                    modified = True
                    print(f"  Sorted HSIresults: {original_order} -> {sorted_order}")
        
        # Sort VNIRresults
        vnir_results = root.find('.//VNIRresults')
        if vnir_results is not None:
            vnir_list = list(vnir_results.findall('HSIresult'))
            if len(vnir_list) > 1:
                vnir_with_pos = []
                for vnir in vnir_list:
                    start_elem = vnir.find('startTime/position')
                    if start_elem is not None:
                        pos = float(start_elem.text)
                        vnir_with_pos.append((pos, vnir))
                
                vnir_with_pos.sort(key=lambda x: x[0])
                
                original_order = [float(vnir.find('startTime/position').text) for vnir in vnir_list if vnir.find('startTime/position') is not None]
                sorted_order = [x[0] for x in vnir_with_pos]
                
                if original_order != sorted_order:
                    for vnir in vnir_list:
                        vnir_results.remove(vnir)
                    for _, vnir in vnir_with_pos:
                        vnir_results.append(vnir)
                    modified = True
                    print(f"  Sorted VNIRresults: {original_order} -> {sorted_order}")
        
        if modified:
            tree.write(xml_path, encoding="utf-8", xml_declaration=True)
            print(f"  Saved sorted results to {os.path.basename(xml_path)}")
        
        return modified
        
    except Exception as e:
        print(f"Error sorting results in {xml_path}: {e}")
        return False

def fix_overlap_depth(xml_path, new_stop_position, old_stop_position):
    """Fix the last piece's stop position in both HSI and VNIR results"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        modified = False
        
        # Fix HSIresults - get last result
        hsi_results = root.find('.//HSIresults')
        if hsi_results is not None:
            hsi_list = list(hsi_results.findall('HSIresult'))
            if hsi_list:
                last_hsi = hsi_list[-1]
                stop_elem = last_hsi.find('stopTime/position')
                if stop_elem is not None:
                    old_stop = float(stop_elem.text)
                    stop_elem.text = f"{new_stop_position:.6f}"
                    print(f"  HSI last piece: stop changed from {old_stop:.6f} to {new_stop_position:.6f}")
                    modified = True
        
        # Fix VNIRresults - get last result
        vnir_results = root.find('.//VNIRresults')
        if vnir_results is not None:
            vnir_list = list(vnir_results.findall('HSIresult'))
            if vnir_list:
                last_vnir = vnir_list[-1]
                stop_elem = last_vnir.find('stopTime/position')
                if stop_elem is not None:
                    old_stop = float(stop_elem.text)
                    stop_elem.text = f"{new_stop_position:.6f}"
                    print(f"  VNIR last piece: stop changed from {old_stop:.6f} to {new_stop_position:.6f}")
                    modified = True
        
        if modified:
            tree.write(xml_path, encoding="utf-8", xml_declaration=True)
            print(f"  ✓ Saved changes to {os.path.basename(xml_path)}")
            return True, old_stop_position
        else:
            print(f"  Warning: No changes made to {os.path.basename(xml_path)}")
            return False, None
            
    except Exception as e:
        print(f"Error fixing overlap in {xml_path}: {e}")
        return False, None

def update_hdr_file_overlap(hdr_path, old_start, old_stop, new_start, new_stop):
    """Update Depth From and Depth To in .hdr file"""
    try:
        with open(hdr_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Update Depth From
        content = re.sub(
            r'Depth From\s*=\s*[0-9.+-]+',
            f'Depth From = {new_start:.6f}',
            content,
            flags=re.IGNORECASE
        )
        
        # Update Depth To
        content = re.sub(
            r'Depth To\s*=\s*[0-9.+-]+',
            f'Depth To = {new_stop:.6f}',
            content,
            flags=re.IGNORECASE
        )
        
        with open(hdr_path, 'w', encoding='utf-8') as f:
            f.write(content)
        
        print(f"    Updated .hdr: {old_start:.6f}-{old_stop:.6f} -> {new_start:.6f}-{new_stop:.6f}")
        return True
        
    except Exception as e:
        print(f"    Error updating .hdr file {hdr_path}: {e}")
        return False

def rename_hsi_files_overlap(hsi_base_dir, measurement_id, old_stop, new_stop):
    """Rename HSI files and update .hdr content when depth changes"""
    try:
        hsi_path = Path(hsi_base_dir)
        if not hsi_path.exists():
            print(f"  Warning: HSI directory not found: {hsi_base_dir}")
            return False
        
        # Look for folders containing the measurement ID (UUID)
        matching_folders = []
        for folder in hsi_path.iterdir():
            if folder.is_dir() and measurement_id in folder.name:
                matching_folders.append(folder)
        
        if not matching_folders:
            print(f"  Warning: No HSI folders found for measurement ID {measurement_id}")
            return False
        
        print(f"  Found {len(matching_folders)} HSI folder(s) to update")
        
        for folder in matching_folders:
            print(f"  Processing folder: {folder.name}")
            
            # Find files with the old stop position
            old_stop_str = f"{old_stop:.3f}"
            new_stop_str = f"{new_stop:.3f}"
            
            files_to_rename = []
            for file in folder.iterdir():
                if file.suffix.lower() in ['.hdr', '.img']:
                    # Check if filename contains the old stop position
                    match = re.search(r'(\d+\.\d+)_' + re.escape(old_stop_str), file.name)
                    if match:
                        start_pos = float(match.group(1))
                        files_to_rename.append((file, start_pos))
            
            if not files_to_rename:
                print(f"    No files found with stop position {old_stop_str}")
                continue
            
            # Process each file
            for file, start_pos in files_to_rename:
                # Create new filename with updated stop position
                new_filename = file.name.replace(
                    f"{start_pos:.3f}_{old_stop_str}",
                    f"{start_pos:.3f}_{new_stop_str}"
                )
                new_filepath = folder / new_filename
                
                # If it's a .hdr file, update content first
                if file.suffix.lower() == '.hdr':
                    update_hdr_file_overlap(file, start_pos, old_stop, start_pos, new_stop)
                
                # Rename the file
                if file != new_filepath:
                    file.rename(new_filepath)
                    print(f"    Renamed: {file.name} -> {new_filename}")
        
        return True
        
    except Exception as e:
        print(f"  Error renaming HSI files: {e}")
        return False

def adjust_depth_overlap():
    """Main function to correct depth overlaps between core boxes"""
    base_folder = folder_entry.get().strip()
    if not base_folder or not os.path.isdir(base_folder):
        messagebox.showerror("Error", "Please provide a valid ANCPRJ folder path.")
        return
    
    # Automatically locate XML and HSI directories
    xml_directory = os.path.join(base_folder, "XML")
    hsi_directory = os.path.join(base_folder, "Masked_Coreg_HSI")
    
    # Validate directories exist
    if not os.path.exists(xml_directory):
        messagebox.showerror("Error", f"XML directory not found: {xml_directory}")
        return
    
    if not os.path.exists(hsi_directory):
        messagebox.showerror("Error", f"Masked_Coreg_HSI directory not found: {hsi_directory}")
        return
    
    xml_path = Path(xml_directory)
    
    old_stdout = sys.stdout
    sys.stdout = TextRedirector(log_text, "stdout")
    
    try:
        print("DEPTH OVERLAP CORRECTION")
        print(f"\nXML Directory: {xml_directory}")
        print(f"HSI Directory: {hsi_directory}\n")
        
        # Step 1: Parse all XML files
        print("Step 1: Parsing XML files...")
        sys.stdout.flush()
        xml_files = list(xml_path.glob("*.xml"))
        
        corebox_data = []
        for xml_file in xml_files:
            info = parse_corebox_info_overlap(xml_file)
            if info:
                corebox_data.append(info)
        
        if not corebox_data:
            print("No valid XML files found!")
            messagebox.showinfo("Info", "No valid XML files found to process.")
            return
        
        # Sort by CoreBox number
        corebox_data.sort(key=lambda x: x['corebox'])
        
        print(f"\nFound {len(corebox_data)} core boxes:")
        for data in corebox_data:
            print(f"  CoreBox {data['corebox']:2d}: {data['first_start']:.6f} to {data['last_stop']:.6f} ({os.path.basename(data['path'])})")
        sys.stdout.flush()
        
        # Step 2: Sort HSI results within each XML
        print(f"\nStep 2: Sorting HSI/VNIR results within each XML...")
        sys.stdout.flush()
        for data in corebox_data:
            print(f"\nProcessing CoreBox {data['corebox']}:")
            sys.stdout.flush()
            sort_hsi_results_in_xml_overlap(data['path'])
        
        # Re-parse after sorting
        print(f"\nRe-parsing XMLs after sorting...")
        sys.stdout.flush()
        corebox_data = []
        for xml_file in xml_files:
            info = parse_corebox_info_overlap(xml_file)
            if info:
                corebox_data.append(info)
        corebox_data.sort(key=lambda x: x['corebox'])
        
        # Step 3: Check for overlaps
        print("Step 3: Checking for overlaps between consecutive boxes...")
        sys.stdout.flush()
        
        overlaps_found = []
        
        for i in range(len(corebox_data) - 1):
            current_box = corebox_data[i]
            next_box = corebox_data[i + 1]
            
            current_end = current_box['last_stop']
            next_start = next_box['first_start']
            
            print(f"\nCoreBox {current_box['corebox']} -> CoreBox {next_box['corebox']}:")
            print(f"  Box {current_box['corebox']} ends at:   {current_end:.6f}")
            print(f"  Box {next_box['corebox']} starts at: {next_start:.6f}")
            sys.stdout.flush()
            
            if current_end > next_start:
                overlap = current_end - next_start
                print(f"  ⚠ OVERLAP DETECTED: {overlap:.6f} meters")
                sys.stdout.flush()
                overlaps_found.append({
                    'current': current_box,
                    'next': next_box,
                    'overlap': overlap
                })
            else:
                gap = next_start - current_end
                print(f"  ✓ OK (gap: {gap:.6f} meters)")
                sys.stdout.flush()
        
        # Step 4: Fix overlaps
        if overlaps_found:
            print(f"Step 4: Fixing {len(overlaps_found)} overlap(s)...")
            sys.stdout.flush()
            
            for overlap_info in overlaps_found:
                current_box = overlap_info['current']
                next_box = overlap_info['next']
                new_stop = next_box['first_start']
                old_stop = current_box['last_stop']
                
                print(f"\nFixing CoreBox {current_box['corebox']}:")
                print(f"  Old stop: {old_stop:.6f}")
                print(f"  New stop: {new_stop:.6f}")
                sys.stdout.flush()
                
                success, actual_old_stop = fix_overlap_depth(current_box['path'], new_stop, old_stop)
                
                if success and current_box['measurement_id']:
                    # Update HSI files using only the UUID
                    rename_hsi_files_overlap(hsi_directory, current_box['measurement_id'], 
                                   old_stop, new_stop)
        else:
            print("No overlaps found! All core boxes are properly aligned.")
            sys.stdout.flush()
        
        print("PROCESSING COMPLETE!")
        
        sys.stdout.flush()
        
        messagebox.showinfo("Success", f"Depth overlap correction completed.\n{len(overlaps_found)} overlaps fixed.")
    
    except Exception as e:
        error_msg = f"An error occurred during depth overlap correction:\n{str(e)}"
        print(error_msg)
        sys.stdout.flush()
        messagebox.showerror("Error", error_msg)
    
    finally:
        sys.stdout = old_stdout



#  TKINTER GUI SETUP   
root= tk.Tk()
root.title("Ancorelog Processing Software")

if hasattr(sys, '_MEIPASS'):
    theme_path = os.path.join(sys._MEIPASS,'tkinter_themes','Azure-ttk-theme-main','azure.tcl')
else:
    theme_path = THEME_PATH_FALLBACK
root.tk.call('source', theme_path)
root.tk.call("set_theme","dark")

ttk.Label(root, text="Folder Path of ancprj:").grid(row=0, column=0, padx=20, pady=10, sticky='e')
folder_entry = ttk.Entry(root, width=50)
folder_entry.grid(row=0, column=1, padx=20, pady=10)
ttk.Button(root, text="Browse", style='Accent.TButton',
           command=lambda: folder_entry.delete(0, tk.END) or folder_entry.insert(0, filedialog.askdirectory())
          ).grid(row=0, column=2, padx=20, pady=10)

ttk.Label(root, text="XRF Calibration File:").grid(row=1, column=0, padx=20, pady=10, sticky='e')
xrfcalib_entry = ttk.Entry(root, width=50)
xrfcalib_entry.grid(row=1, column=1, padx=20, pady=10)
ttk.Button(root, text="Browse", style='Accent.TButton',
           command=lambda: xrfcalib_entry.delete(0, tk.END) or xrfcalib_entry.insert(0, filedialog.askopenfilename(filetypes=[("Text files","*.txt")]))
          ).grid(row=1, column=2, padx=20, pady=10)

ttk.Label(root, text="XML File location:").grid(row=2, column=0, padx=20, pady=10, sticky='e')
xml_entry = ttk.Entry(root, width=50)
xml_entry.grid(row=2, column=1, padx=20, pady=10)
ttk.Button(root, text="Browse", style='Accent.TButton',
           command=lambda: xml_entry.delete(0, tk.END) or xml_entry.insert(0, filedialog.askopenfilename(filetypes=[("XML files","*.xml")]))
          ).grid(row=2, column=2, padx=20, pady=10)

ttk.Label(root, text="Files to be Masked:").grid(row=3, column=0, padx=20, pady=10, sticky='e')
mask_entry = ttk.Entry(root, width=50)
mask_entry.grid(row=3, column=1, padx=20, pady=10)
ttk.Button(root, text="Browse", style='Accent.TButton',
           command=lambda: mask_entry.delete(0, tk.END) or mask_entry.insert(0, filedialog.askdirectory())
          ).grid(row=3, column=2, padx=20, pady=10)



for col in range(3):
    root.grid_columnconfigure(col, weight=1)

# set up a container for all of the buttons and center it under cols 0–2
button_frame = ttk.Frame(root)
button_frame.grid(row=6, column=0, columnspan=3, pady=10)

# inside the frame, give both internal columns equal weight
button_frame.grid_columnconfigure(0, weight=1)
button_frame.grid_columnconfigure(1, weight=1)

# left-column buttons
ttk.Button(button_frame, text="Extract Files from ANCPRJ",
           style='Accent.TButton', command=rename_and_extract)\
    .grid(row=0, column=0, padx=10, pady=5, sticky='ew')

ttk.Button(button_frame, text="Correct HSI Files",
           style='Accent.TButton', command=reflect_correct_hsi)\
    .grid(row=1, column=0, padx=10, pady=5, sticky='ew')

ttk.Button(button_frame, text="Coregister VNIR/SWIR Data",
           style='Accent.TButton', command=coregister_hsi_optimized)\
    .grid(row=2, column=0, padx=10, pady=5, sticky='ew')

ttk.Button(button_frame, text="Mask HSI Files",
           style='Accent.TButton', command=mask_hsi_files)\
    .grid(row=3, column=0, padx=10, pady=5, sticky='ew')

# right-column buttons

ttk.Button(button_frame, text="Compile XRF Data", 
          style='Accent.TButton', command=start_processing_xml_combine)\
    .grid(row=0, column=1, padx=10, pady=5, sticky='ew')

ttk.Button(button_frame, text="Extract RGB Core Pieces",
           style='Accent.TButton', command=extract_rgb_core_pieces)\
    .grid(row=1, column=1, padx=10, pady=5, sticky='ew')
        
ttk.Button(button_frame, text="Make Depth Overlap Correction",
           style='Accent.TButton', command=adjust_depth_overlap)\
    .grid(row=2, column=1, padx=10, pady=5, sticky='ew')

log_text = tk.Text(root, height=11, width=90)
log_text.grid(row=12, column=0, columnspan=3, padx=10, pady=10)
log_text.insert(tk.END, "Welcome to the Ancorelog Processor!\n")

root.mainloop()